# Lab 21 · Track 3 — Fine-tuning LLM · **Kaggle FULL run** (một notebook duy nhất)

Chạy từ trên xuống. Notebook này **tự chứa toàn bộ harness** — không `git clone`
repo của ai, không tải script từ đâu. Mọi module `labkit` được ghi ra đĩa bằng
`%%writefile` ở phần 3 và bạn đọc được toàn bộ code ngay trong notebook.

### Trước khi chạy
| Cần | Ở đâu |
|---|---|
| **GPU T4** | Settings → Accelerator → **GPU T4 x2** (notebook tự khoá về 1 card) |
| **Internet ON** | Settings → Internet → On (để `pip install` + tải base model) |
| **Dataset `LAB21_VIN`** | Add Input → Datasets → `LAB21_VIN` (5 file `.jsonl`/`.json`) |

### Notebook này trả lời gì (bản đồ sang yêu cầu của lab)
| Yêu cầu | Ở phần |
|---|---|
| Chat template + **mask loss có bằng chứng** | §4 |
| `max_length` lấy từ **p95 đo được** | §4 |
| Ba phiên bản (a) naive · (b) prompt tối ưu · (c) fine-tune, đo **trước** khi train | §6 |
| Train cấu hình đúng (all-linear · LR 10× · batch<32 · alpha=2r) | §7 |
| **Ba cấu hình sai** cùng ngân sách step, ngân sách tham số khớp | §8 |
| Bốn nhóm điểm **target · regression · format · latency** + phán quyết | §9 |
| **Latency & cost** ($/1k ticket, break-even) | §10 |
| **Chạy thử câu hỏi thật**, gồm cả câu ngoài miền | §11 |
| Merge + hoán đổi adapter | §12 |
| Cổng kiểm tra trước khi nộp + REPORT.md + zip artefact | §13 |

### Ba thứ notebook này làm mà pipeline gốc chưa làm
1. **`correct_replay`** — deck §14.3: trộn 1–5% dữ liệu tổng quát đã *chứng minh
   không trùng* tập regression, train ở **đúng cùng số step** với `correct`. Đây là
   đối chứng cho đúng khuyết điểm mà repo tự đo được: regression **0.644 → 0.067**.
2. **Loss trên tập held-out ngay trong lúc train** — val split đã có từ §4 nhưng
   chưa ai dùng; overfit trở thành số đo thay vì suy đoán.
3. **Latency đã trừ warm-up + đếm token** → cost là phép tính trên số đo, và
   "prompt ngắn đi" trở thành một lập luận kinh tế.

## 1. Môi trường — 1 GPU, dependency, không clone gì cả

Kaggle cấp **2×T4**. `device_map="auto"` sẽ chia model qua cả hai card, và một
model bị shard + LoRA + `GradScaler` của fp16 là đường ngắn nhất tới
`expected all tensors on the same device` ở step 0. Khoá về 1 card **trước khi**
torch được import lần đầu — sau khi torch đã khởi tạo CUDA thì đặt biến môi trường
không còn tác dụng.

In [ ]:
import os, pathlib, subprocess, sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # phải đặt TRƯỚC khi import torch
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

WORK = pathlib.Path("/kaggle/working")
if not WORK.exists():
    WORK = pathlib.Path.cwd()
os.chdir(WORK)
for sub in ("data", "results", "adapters", "submission"):
    (WORK / sub).mkdir(exist_ok=True)
print("cwd:", WORK)

# Pin theo requirements.txt của lab. Cài TRƯỚC khi import transformers/trl để
# kernel không phải restart: một notebook đòi restart giữa 3 tiếng train là một
# notebook sẽ bị chạy lại từ đầu.
PKGS = [
    "transformers>=5.15,<6", "trl>=1.10,<2", "peft>=0.20,<1",
    "accelerate>=1.14,<2", "datasets>=5,<6", "torchao>=0.16",
    "tokenizers>=0.22", "bitsandbytes>=0.50",
]
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PKGS],
                      capture_output=True, text=True)
print(proc.stdout[-2000:] or "pip: ok")
if proc.returncode:
    print(proc.stderr[-3000:])
    raise SystemExit(
        "pip install thất bại. Kiểm tra Settings → Internet = On."
    )

import torch, transformers, trl, peft, datasets
print(f"torch {torch.__version__} · transformers {transformers.__version__} · "
      f"trl {trl.__version__} · peft {peft.__version__} · datasets {datasets.__version__}")
if not torch.cuda.is_available():
    raise SystemExit("Không thấy GPU. Settings → Accelerator → GPU T4 x2.")
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} · {p.total_memory/1024**3:.1f} GB · sm_{p.major}{p.minor} "
      f"· visible devices = {torch.cuda.device_count()}")

## 2. CONFIG — mọi nút của cả lab nằm trong ô này

Đọc hết ô này rồi hãy chạy. `EVAL_LIMIT = ""` là bài nộp (full eval); đặt `"8"` để
tổng duyệt. `COMPUTE_TIER = "SMOKE"` đổi sang checkpoint 0.8B để rà đường ống trong
~25 phút — điểm tuyệt đối của SMOKE **không** so được với T4, chỉ có *thứ tự* là
thứ đáng tin ở một lần tổng duyệt.

In [ ]:
# ---- các nút của lab -------------------------------------------------------
COMPUTE_TIER    = "T4"              # "T4" = 4B (bài nộp) | "SMOKE" = 0.8B (tổng duyệt)
MODEL_ID        = ""                # để trống = checkpoint của tier
EVAL_LIMIT      = ""                # "" = FULL (bài nộp) | "8" = smoke
EPOCHS          = "2"               # chung cho MỌI run, nên các run so được
MASK_MODE       = "assistant-only"
REPLAY_FRACTION = "0.05"            # deck §14.3: 1–5%
SEED            = 42

RUN_REPLAY      = True              # §7b: correct_replay (+~25')
RUN_CONTRASTS   = True              # §8: attn_only / wrong_lr / qlora (+~75')
RUN_MERGE       = False             # §12: merge ghi ~8 GB xuống /kaggle/working
FORCE_RETRAIN   = False             # True = train lại cả adapter đã có

# giá dùng cho §10. Đây là GIẢ ĐỊNH, không phải số đo — đổi theo hoá đơn của bạn.
GPU_HOURLY_USD  = 0.35              # T4 spot/preemptible ~2026
API_IN_USD_MTOK = 0.15              # API small-model, $/1M token vào
API_OUT_USD_MTOK = 0.60             # $/1M token ra

# ---- đẩy xuống os.environ TRƯỚC khi import labkit --------------------------
import os
os.environ["COMPUTE_TIER"] = COMPUTE_TIER
os.environ["EPOCHS"] = EPOCHS
os.environ["MASK_MODE"] = MASK_MODE
os.environ["REPLAY_FRACTION"] = REPLAY_FRACTION
os.environ["LAB_RESULTS"] = str(WORK / "results")
if MODEL_ID.strip():
    os.environ["MODEL_ID"] = MODEL_ID.strip()
else:
    os.environ.pop("MODEL_ID", None)
if EVAL_LIMIT.strip():
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT.strip()
else:
    os.environ.pop("EVAL_LIMIT", None)

SMOKE = bool(EVAL_LIMIT.strip()) or COMPUTE_TIER.upper() == "SMOKE"
plan = [
    ("§4  data + mask + split", "~2'"),
    ("§5  replay corpus + decontamination", "~5s"),
    ("§6  baselines (a) và (b)", "~12'"),
    ("§7  train correct", "~25'"),
    ("§7b train correct_replay", "~25'" if RUN_REPLAY else "TẮT"),
    ("§8  3 contrast (attn_only/wrong_lr/qlora)", "~75'" if RUN_CONTRASTS else "TẮT"),
    ("§9  chấm 4 nhóm + phán quyết", "~20'"),
    ("§10 latency + cost", "~0'"),
    ("§11 câu hỏi thử", "đã đo ở §6/§9"),
    ("§12 merge + hot-swap", "~8'" if RUN_MERGE else "TẮT"),
    ("§13 cổng kiểm tra + REPORT + zip", "~1'"),
]
print(f"tier={COMPUTE_TIER}  EVAL_LIMIT={EVAL_LIMIT or 'FULL'}  EPOCHS={EPOCHS}  "
      f"replay={REPLAY_FRACTION}  smoke_mode={SMOKE}")
if SMOKE:
    print("\n⚠ SMOKE MODE — kết quả KHÔNG dùng để nộp (§13 sẽ báo FAIL).")
print("\nKế hoạch (ước lượng trên T4, tier T4/4B):")
for what, how_long in plan:
    print(f"  {what:<44} {how_long}")

## 3. Dữ liệu — tìm dataset đã attach và **kiểm tra checksum**

Bốn file corpus là hợp đồng của bài lab: sửa tập eval sau khi thấy kết quả thì mọi
so sánh phía sau đều vô nghĩa. `checksums.json` đi kèm dataset là cách kiểm việc đó
bằng một con số.

Một chi tiết thật, không phải phòng xa: các checksum ấy được tính trên nội dung
**LF**, còn một checkout Windows (`core.autocrlf=true`) cho ra file **CRLF** — cùng
dữ liệu, khác byte, khác SHA. Nên ô này băm cả hai dạng và chỉ báo FAIL khi *cả hai*
đều lệch. Băm thô một mình sẽ tố oan mọi dataset upload từ Windows.

In [ ]:
import hashlib, json, pathlib, shutil

NEEDED = ["train_seed.jsonl", "eval_target.jsonl",
          "eval_regression.jsonl", "holdout_secret.jsonl"]
DATA = WORK / "data"

def _find_dataset_root() -> pathlib.Path:
    roots = [p for p in pathlib.Path("/kaggle/input").glob("*")] \
            if pathlib.Path("/kaggle/input").exists() else []
    cands = []
    for root in roots:
        for d in [root, *(p for p in root.rglob("*") if p.is_dir())]:
            if (d / "train_seed.jsonl").exists():
                cands.append(d)
    if not cands:
        if all((DATA / f).exists() for f in NEEDED):
            return DATA           # đã copy từ lần chạy trước
        raise SystemExit(
            "Không tìm thấy train_seed.jsonl trong /kaggle/input.\n"
            "Add Input → Datasets → LAB21_VIN rồi chạy lại ô này."
        )
    # ưu tiên thư mục có đủ nhất
    cands.sort(key=lambda d: sum((d / f).exists() for f in NEEDED), reverse=True)
    return cands[0]

SRC_DATA = _find_dataset_root()
print("dataset:", SRC_DATA)
for f in NEEDED + ["checksums.json"]:
    s = SRC_DATA / f
    if s.exists():
        if s.resolve() != (DATA / f).resolve():
            shutil.copy2(s, DATA / f)
    elif f in NEEDED:
        raise SystemExit(f"thiếu {f} trong dataset đã attach")
    else:
        print(f"  (không có {f} — bỏ qua bước kiểm checksum)")

def sha16(path: pathlib.Path) -> tuple[str, str]:
    """(sha thô, sha sau khi chuẩn hoá CRLF->LF), mỗi cái 16 hex đầu."""
    b = path.read_bytes()
    return (hashlib.sha256(b).hexdigest()[:16],
            hashlib.sha256(b.replace(b"\r\n", b"\n")).hexdigest()[:16])

integrity = {"dataset_dir": str(SRC_DATA), "files": {}, "drift": []}
ref = {}
if (DATA / "checksums.json").exists():
    ref = json.loads((DATA / "checksums.json").read_text(encoding="utf-8"))
for f in NEEDED:
    p = DATA / f
    raw, lf = sha16(p)
    n = sum(1 for line in p.open(encoding="utf-8") if line.strip())
    want = ref.get(f)
    ok = want is None or want in (raw, lf)
    integrity["files"][f] = {"rows": n, "sha_raw": raw, "sha_lf": lf,
                             "expected": want, "ok": ok}
    if not ok:
        integrity["drift"].append(f)
    flag = "ok" if ok else "DRIFT"
    print(f"  {f:<24} {n:>4} dòng  sha={raw} lf={lf}  expected={want}  [{flag}]")

if integrity["drift"]:
    raise SystemExit(
        f"checksum lệch: {integrity['drift']}. Tập eval đã bị sửa — mọi so sánh "
        "sau đây sẽ vô nghĩa. Attach lại dataset gốc."
    )
print("\ncorpus khớp checksum.")

## 4. Harness `labkit` — ghi thẳng ra đĩa, không clone

Mười một module dưới đây **là** thư viện của lab. Chúng ở đây dưới dạng
`%%writefile` vì luật của bài: notebook Kaggle không được đi clone repo của bên
khác. Đổi lại, bạn đọc được toàn bộ code đang chạy — kể cả những chỗ khó, như vì sao
mask được dựng từ **offset ký tự** chứ không phải bằng cách trừ token
(`data.build_example`), hay vì sao `assistant_only_loss` của TRL bị **cố ý không
dùng** (`train.sft_config_kwargs`).

In [ ]:
import pathlib
pathlib.Path("labkit").mkdir(exist_ok=True)
print("labkit/ ready")

In [ ]:
%%writefile labkit/__init__.py
"""labkit (Kaggle edition) — the shared harness for Day 21's fine-tuning lab.

Same contract as `src/labkit`, with the pipeline improvements the Kaggle notebook
adds on top. Differences from the repo copy are listed in one place so the notebook
can print them and the report can cite them:

    KAGGLE_DELTAS  -- what changed, and why

No `.env` loading here: on Kaggle every knob comes from the CONFIG cell, which sets
`os.environ` before this package is imported.
"""

KAGGLE_DELTAS = [
    ("replay mix (§14.3)",
     "data.mix_replay() folds a decontaminated general-instruction corpus into the "
     "training set, and `correct_replay` trains on it at the SAME step budget as "
     "`correct`. This targets the one measured, unfixed defect in the repo pipeline: "
     "regression 0.644 -> 0.067, a total capability collapse."),
    ("held-out loss during training",
     "train.sft_config_kwargs() wires data/split/val.jsonl in as eval_dataset, so "
     "overfitting is visible in the log instead of inferred after the fact."),
    ("F-31 guard is an assert, not a docstring",
     "data.assert_prompt_alignment() fails the run when the evaluation render is not "
     "a prefix of the training render — the defect that made every adapter score 0.000."),
    ("latency excludes warm-up",
     "generate.generate_batch(warmup=True) discards a throwaway generation before "
     "timing, and returns token counts, so ms/sample and ms/token are both real."),
    ("cost is measured, not asserted",
     "cost.py turns measured latency + prompt length into $/1k tickets for each "
     "version, which is what makes 'the prompt can shrink' an economic claim."),
    ("holdout_secret.jsonl is finally used",
     "the shipped 20-item holdout is scored once, on the winning version only, as a "
     "check against having tuned on eval_target."),
    ("integrity checks run inside the notebook",
     "checksums of the four corpus files, the SHA of prompt (b), the shared step "
     "budget and the matched parameter budget are all asserted in-line rather than "
     "by a separate verify script the grader has to trust was run."),
]

__all__ = ["config", "cost", "data", "evaluate", "generate", "modeling", "replay",
           "report", "train", "device", "KAGGLE_DELTAS"]
__version__ = "1.0.0-kaggle"

In [ ]:
%%writefile labkit/config.py
"""Tier + model resolution for Lab 21 (Kaggle edition).

One source of truth for "which model on which GPU". Everything else imports from here
so a student changes `COMPUTE_TIER` in the notebook's CONFIG cell and the whole lab
follows.

Kaggle deltas:
  * a `SMOKE` tier — the 0.8B checkpoint at T4 sequence settings, so the whole
    notebook can be rehearsed end to end in ~25 minutes before the 4B run is
    committed to. The repo's `CPU` tier is 0.8B too, but at `max_length=512`, which
    truncates differently and therefore is not a rehearsal of the graded path.
  * `$MODEL_ID` overrides the tier's checkpoint. Kaggle sessions die, quotas run
    out, and hub IDs move; a run that cannot substitute a checkpoint without editing
    library code is a run that cannot be finished.
  * a `correct_replay` LoRA spec — same knobs as `correct`, different *data*. See
    `labkit.replay`.

Design note (deck §12): Unsloth's own Qwen3.5 guidance is *do not use QLoRA on this
generation* — quantization error is higher than normal. So the default path here is
**bf16 LoRA**, and 4-bit is something the student *measures* in NB4 rather than
assumes. That is the opposite of the 2024-era lab this replaces.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Tier:
    name: str
    model_id: str
    vram_gb_bf16_lora: float
    max_length: int
    per_device_batch: int
    grad_accum: int
    notes: str

    @property
    def effective_batch(self) -> int:
        return self.per_device_batch * self.grad_accum


# VRAM figures are the vendor's published bf16-LoRA minimums for the Qwen3.5 family
# (0.8B→3GB, 2B→5GB, 4B→10GB, 9B→22GB, 27B→56GB). Keep them honest: if you change the
# model, change the number, and re-measure with `torch.cuda.max_memory_allocated()`.
TIERS: dict[str, Tier] = {
    "CPU": Tier(
        name="CPU",
        model_id="Qwen/Qwen3.5-0.8B",
        vram_gb_bf16_lora=0.0,
        max_length=512,
        per_device_batch=1,
        grad_accum=8,
        notes="No GPU. NB1 (data+mask) and the eval harness run here; training does not.",
    ),
    "LAPTOP": Tier(
        name="LAPTOP",
        model_id="Qwen/Qwen3.5-2B",
        vram_gb_bf16_lora=5.0,
        max_length=1024,
        per_device_batch=1,
        grad_accum=8,
        notes="8-12 GB laptop GPU (RTX 3060/4060).",
    ),
    "T4": Tier(
        name="T4",
        model_id="unsloth/Qwen3.5-4B",
        vram_gb_bf16_lora=10.0,
        max_length=1024,
        per_device_batch=1,
        grad_accum=16,
        notes="Free Colab T4 / Kaggle T4 (16 GB) — the default path for this lab.",
    ),
    # Kaggle addition. Not a hardware tier: the same T4, deliberately under-modelled so
    # the notebook can be rehearsed end to end before ~3 hours of GPU quota is spent on
    # the 4B run. Absolute scores from this tier are NOT comparable to T4's; the
    # ORDERINGS are what a rehearsal is for.
    "SMOKE": Tier(
        name="SMOKE",
        model_id="Qwen/Qwen3.5-0.8B",
        vram_gb_bf16_lora=3.0,
        max_length=1024,
        per_device_batch=1,
        grad_accum=16,
        notes="Rehearsal tier: 0.8B at T4 sequence settings. Fast, not submittable.",
    ),
    "BIGGPU": Tier(
        name="BIGGPU",
        model_id="Qwen/Qwen3.5-9B",
        vram_gb_bf16_lora=22.0,
        max_length=2048,
        per_device_batch=2,
        grad_accum=8,
        notes="L4 22.5 GB / A100 40 GB / RTX 3090-4090.",
    ),
}

DEFAULT_TIER = "T4"


def get_tier(name: str | None = None) -> Tier:
    """Resolve the active tier from an argument, then $COMPUTE_TIER, then the default.

    `$MODEL_ID`, when set, replaces the tier's checkpoint and leaves every other knob
    alone. That is deliberate: sequence length, batch and grad-accum were chosen for the
    *card*, not for the checkpoint, so swapping the model should not silently re-tune
    them. The substitution is recorded in `results/run_config.json` so a report cannot
    quietly claim a 4B number from a 0.8B run.
    """
    key = (name or os.environ.get("COMPUTE_TIER") or DEFAULT_TIER).upper()
    if key not in TIERS:
        raise ValueError(
            f"Unknown COMPUTE_TIER={key!r}. Pick one of: {', '.join(TIERS)}"
        )
    tier = TIERS[key]
    override = os.environ.get("MODEL_ID", "").strip()
    if override and override != tier.model_id:
        from dataclasses import replace
        tier = replace(tier, model_id=override,
                       notes=tier.notes + f" [MODEL_ID override: {override}]")
    return tier


# --- Training configuration -------------------------------------------------
# The deck's §10 result: the learning rate should be set on the *scale* of ~10x the
# full-fine-tune LR, and that scale matters more than the exact value. A full FT of a
# 4B model sits near 1e-5, so LoRA lands near 1e-4.
FULL_FT_LR = 1e-5
LORA_LR_MULTIPLIER = 10.0
LORA_LR = FULL_FT_LR * LORA_LR_MULTIPLIER          # 1e-4

# §10.4: LoRA tolerates large batches worse than full FT, and raising rank does not
# fix it. Keep the effective batch under 32.
MAX_EFFECTIVE_BATCH = 32


@dataclass(frozen=True)
class LoraSpec:
    """A named LoRA configuration. NB3 trains `correct`; NB4 trains the rest as contrasts."""
    key: str
    r: int | None          # None => computed to match `correct`'s parameter budget
    alpha: int | None      # None => set to 2*r once r is known
    target: str               # "text-linear" | "attn-only"
    lr: float
    load_in_4bit: bool
    label: str
    teaches: str

    @property
    def alpha_over_r(self) -> float | None:
        if self.r is None or self.alpha is None:
            return None
        return self.alpha / self.r

    def resolved(self, r: int) -> "LoraSpec":
        """Fill in a computed rank (alpha follows the deck's 2r invariant, §9.3)."""
        from dataclasses import replace
        return replace(self, r=r, alpha=2 * r)


SPECS: dict[str, LoraSpec] = {
    "correct": LoraSpec(
        key="correct", r=16, alpha=32, target="text-linear", lr=LORA_LR,
        load_in_4bit=False,
        label="all-linear · r=16 · LR 10x · 16-bit",
        teaches="The deck's low-regret configuration (§10).",
    ),
    # r=None => resolved at runtime by modeling.matched_rank() so this run sits on the
    # SAME trainable-parameter budget as `correct`. On Qwen3.5-4B that lands near r=90.
    # Hardcoding a rank here would compare budgets instead of placements.
    "attn_only": LoraSpec(
        key="attn_only", r=None, alpha=None, target="attn-only", lr=LORA_LR,
        load_in_4bit=False,
        label="q,v only · r=matched · LR 10x · 16-bit",
        teaches="Mistake #1 (§10.2): attention-only placement, rank raised to *match "
                "parameter count*. If rank were the lever, this would win.",
    ),
    "wrong_lr": LoraSpec(
        key="wrong_lr", r=16, alpha=32, target="text-linear", lr=FULL_FT_LR,
        load_in_4bit=False,
        label="all-linear · r=16 · LR 1x (full-FT scale) · 16-bit",
        teaches="Mistake #2 (§10.3): a full-fine-tune learning rate applied to LoRA.",
    ),
    "qlora": LoraSpec(
        key="qlora", r=16, alpha=32, target="text-linear", lr=LORA_LR,
        load_in_4bit=True,
        label="all-linear · r=16 · LR 10x · 4-bit QLoRA",
        teaches="The vendor says do NOT use QLoRA on Qwen3.5 (§12). Measure the cost "
                "yourself instead of taking either side on faith.",
    ),
    # --- Kaggle addition -----------------------------------------------------------
    # Every LoRA knob is IDENTICAL to `correct`. The only variable is the training
    # corpus: `correct` trains on triage tickets alone, `correct_replay` trains on the
    # same tickets plus a small decontaminated slice of general instructions.
    #
    # This is the one contrast the repo pipeline is missing, and it is aimed at the one
    # defect the repo's own findings measured and left open: the fine-tune's general
    # capability fell from 0.644 to 0.067 because a corpus of nothing but
    # `ticket -> JSON` teaches the model that ANY input means "emit triage JSON".
    # Deck §14.3 names the remedy (1-5% replay); nothing in the lab tested it.
    "correct_replay": LoraSpec(
        key="correct_replay", r=16, alpha=32, target="text-linear", lr=LORA_LR,
        load_in_4bit=False,
        label="all-linear · r=16 · LR 10x · 16-bit · + replay mix",
        teaches="Deck §14.3: the anti-forgetting remedy, as a controlled contrast. One "
                "variable versus `correct` — the DATA, not the LoRA configuration.",
    ),
}


# Fraction of the mixed training corpus that is replay (general-instruction) data.
# Deck §14.3 says 1-5%. The measured collapse on this corpus is total rather than
# partial, so the default sits at the top of that band and is a knob, not a constant:
# REPLAY_FRACTION is one of the things the report is expected to have an opinion about.
REPLAY_FRACTION_DEFAULT = 0.05


def replay_fraction(override: float | None = None) -> float:
    if override is not None:
        return float(override)
    raw = os.environ.get("REPLAY_FRACTION", "").strip()
    frac = float(raw) if raw else REPLAY_FRACTION_DEFAULT
    if not 0.0 <= frac < 0.5:
        raise ValueError(
            f"REPLAY_FRACTION={frac} is outside [0, 0.5). Above ~0.2 you are no longer "
            "running an anti-forgetting mix, you are training a different task."
        )
    return frac


# --- prompts -----------------------------------------------------------------------
# These live here, not in generate.py, because BOTH training and evaluation need them
# and holding two copies is how F-31 happened: the lab trained on one prompt shape and
# scored on another, and every adapter came out at target=0.000.
NAIVE_PROMPT = "Phân loại ticket sau."

OPTIMIZED_PROMPT = """Bạn là hệ thống phân loại ticket CSKH. Trả về DUY NHẤT một object JSON, không kèm giải thích, không kèm markdown fence.

Schema bắt buộc — đúng 4 khóa:
{"intent": ..., "urgency": ..., "product": ..., "sentiment": ...}

intent    ∈ doi_tra | van_chuyen | hoan_tien | san_pham_loi | hoi_thong_tin
urgency   ∈ cao | trung_binh | thap
sentiment ∈ tieu_cuc | trung_tinh | tich_cuc
product   = tên sản phẩm xuất hiện nguyên văn trong ticket

Ví dụ:
Ticket: "Shop ơi, mình đặt bàn phím cơ mã đơn DH123456. Giao hàng chậm. Đã 3 ngày rồi. Nhờ shop kiểm tra."
JSON: {"intent": "van_chuyen", "urgency": "trung_binh", "product": "bàn phím cơ", "sentiment": "trung_tinh"}"""


CONTRAST_KEYS = ["attn_only", "wrong_lr", "qlora"]

# Every run that must land on the SAME optimizer-step budget to be comparable. The
# replay run is in here for the same reason the misconfig contrasts are: it differs from
# `correct` in exactly one variable, and the step count is not allowed to be a second
# one. Note that a replay mix makes the corpus LARGER, so `correct_replay` cannot derive
# its step count from its own length — it has to be handed `correct`'s.
GRADED_KEYS = ["correct", "correct_replay", *CONTRAST_KEYS]

EPOCHS_DEFAULT = 2.0


def training_epochs(override: float | None = None) -> float:
    """The epoch budget -- ONE number, shared by NB3's baseline and NB4's contrasts.

    NB4's contrasts must run the SAME number of optimizer steps as NB3's `correct` run,
    otherwise the autopsy measures step budget instead of configuration. Both notebooks
    turn this into a step count with `train.planned_steps(n_examples, tier, epochs)`.

    Why a function and not two constants. This started as `CONTRAST_MAX_STEPS = 60`,
    calibrated against a "~10 minutes" estimate; measured on a free-Colab T4 at 48.5
    s/step that is 48 minutes per contrast, and 2x the 30 steps NB3 actually runs -- so
    every contrast was trained twice as long as the baseline it is compared against.
    The first fix made both notebooks derive the count, but left NB3 reading $EPOCHS
    while NB4 read a frozen constant: setting `EPOCHS=1` re-opened the same divergence
    through the front door. One reader, one env var, no way to set half of it.

    `EPOCHS=1` is the supported lever when you are time-boxed -- it halves NB3 *and*
    NB4. Submit with the default, and `scripts/verify.py` cross-checks the step budget
    recorded in `results/runs.csv` so the fairness of the comparison is a checked fact
    rather than a promise.
    """
    if override is not None:
        return float(override)
    raw = os.environ.get("EPOCHS", "").strip()
    return float(raw) if raw else EPOCHS_DEFAULT

In [ ]:
%%writefile labkit/device.py
"""Device and precision selection.

**Why this file exists.** The lab's default tier is a free-Colab **T4**, and a T4 is
Turing (compute capability 7.5). **Turing has no bfloat16 support** — bf16 needs
Ampere (8.0) or newer. Hardcoding `bf16=True`, which every 2026 tutorial does because
every 2026 tutorial is written on an A100, is wrong on the exact hardware this lab
tells students to use.

fp16 is the correct choice there, and it is not merely a flag swap: fp16 has a much
smaller exponent range, so training needs **gradient scaling** to avoid underflow.
Trainers enable a GradScaler automatically when told `fp16=True`, which is why the
precision decision must reach the training arguments rather than only the model load.
"""
from __future__ import annotations


def describe() -> dict:
    """What we are actually running on. Print this before training."""
    info = {"device": "cpu", "name": "CPU", "bf16": False, "fp16": False,
            "capability": None, "vram_gb": None}
    try:
        import torch
    except ImportError:                                   # pragma: no cover
        return info

    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        # DO NOT use torch.cuda.is_bf16_supported() bare. Its signature is
        # `is_bf16_supported(including_emulation: bool = True)`, and with the default it
        # returns True on a T4 — it merely checks that a bf16 tensor can be *created*,
        # which Turing can do by emulation. Training then "works" at a large speed
        # penalty while reporting bf16=True. Hardware support is compute capability
        # >= 8.0 (Ampere), full stop.
        native_bf16 = major >= 8
        try:                       # belt and braces on builds that expose the kwarg
            native_bf16 = native_bf16 and torch.cuda.is_bf16_supported(
                including_emulation=False)
        except TypeError:          # older torch without the kwarg
            pass
        info.update(
            device="cuda",
            name=torch.cuda.get_device_name(0),
            capability=f"{major}.{minor}",
            vram_gb=round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 1),
            bf16=bool(native_bf16),
            fp16=True,
        )
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        # MPS: fp16 works; bf16 support is patchy across torch versions, so do not
        # claim it. Training here is for smoke-testing the code path, not for results.
        info.update(device="mps", name="Apple MPS", fp16=True, bf16=False)
    return info


def precision(explicit: str | None = None) -> str:
    """Return 'bf16' | 'fp16' | 'fp32' for the current device."""
    if explicit:
        if explicit not in ("bf16", "fp16", "fp32"):
            raise ValueError(f"precision must be bf16|fp16|fp32, got {explicit!r}")
        return explicit
    info = describe()
    if info["bf16"]:
        return "bf16"
    if info["fp16"]:
        return "fp16"
    return "fp32"


def torch_dtype(explicit: str | None = None):
    """The torch dtype matching `precision()`."""
    import torch
    return {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}[
        precision(explicit)
    ]


def banner() -> str:
    info = describe()
    p = precision()
    line = f"{info['name']} ({info['device']}"
    if info["capability"]:
        line += f", sm_{info['capability'].replace('.', '')}"
    if info["vram_gb"]:
        line += f", {info['vram_gb']} GB"
    line += f") -> precision={p}"
    if info["device"] == "cuda" and not info["bf16"]:
        line += ("\n  note: this GPU predates Ampere, so it has NO bfloat16. Using fp16 "
                 "with gradient scaling instead. Tutorials that hardcode bf16=True fail here.")
    return line

def has_flash_attention() -> bool:
    """Is a FlashAttention kernel importable?

    TRL warns that `padding_free=True` is only known to behave with
    flash_attention_2/3 kernels. FlashAttention-2 needs **Ampere (sm_80) or newer**, so
    on the lab's default T4 it is unavailable no matter what you install.
    """
    for mod in ("flash_attn", "kernels"):
        try:
            __import__(mod)
            return True
        except ImportError:
            continue
    return False


def supports_padding_free(min_batch: int = 2) -> bool:
    """Whether `padding_free=True` is both SAFE and USEFUL here.

    Two independent conditions, and the lab's default tier fails both:

    * **Safe** — needs a FlashAttention kernel. Padding-free flattens a batch into one
      sequence; without a kernel that understands the boundaries, attention can run
      across them. That is exactly deck §13.3's warning ("packing is free only when
      sequence boundaries are respected") applied to its sibling flag.
    * **Useful** — pointless at `per_device_train_batch_size=1`: there is no padding
      between sequences to remove when a batch holds one sequence. TRL says so
      outright: *"Using a batch size of 1 annihilates the benefits of padding-free."*
    """
    return has_flash_attention() and min_batch >= 2

In [ ]:
%%writefile labkit/report.py
"""Results I/O. Every run appends one row with the same columns, so the report table
writes itself and the grader can verify your numbers are internally consistent.

Kaggle deltas: the results directory is resolved from `$LAB_RESULTS` (falling back to
`./results`) instead of from this file's location. The repo copy computes
`parents[2]/"results"`, which is correct for `src/labkit/report.py` inside a checkout
and wrong for `/kaggle/working/labkit/report.py` — there it resolves to `/results`,
which is not writable, and the failure appears at the end of a 3-hour run. Plus
`hbar()`, so score comparisons are visible at a glance without a plotting dependency.
"""
from __future__ import annotations

import csv
import json
import os
import pathlib
from typing import Iterable


def results_dir_default() -> pathlib.Path:
    env = os.environ.get("LAB_RESULTS", "").strip()
    if env:
        return pathlib.Path(env)
    here = pathlib.Path(__file__).resolve()
    # In a repo checkout (`<root>/src/labkit/report.py` or
    # `<root>/colab/kaggle_src/labkit/report.py`) prefer the checkout's results dir.
    for parent in here.parents:
        if (parent / "data").is_dir() and (parent / "requirements.txt").exists():
            return parent / "results"
    return pathlib.Path.cwd() / "results"


RESULTS = results_dir_default()


def append_row(row: dict, filename: str = "runs.csv", results_dir: pathlib.Path | None = None) -> pathlib.Path:
    """Append `row`, creating the header from its keys on first write.

    Rows with new keys are unioned into the header rather than silently dropped —
    losing a column because run 3 measured something run 1 did not is exactly the kind
    of quiet data loss that makes a results table untrustworthy.
    """
    out_dir = results_dir or results_dir_default()
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / filename

    existing: list[dict] = []
    if path.exists():
        with path.open(encoding="utf-8", newline="") as fh:
            existing = list(csv.DictReader(fh))

    rows = existing + [row]
    fields: list[str] = []
    for r in rows:
        for k in r:
            if k not in fields:
                fields.append(k)

    with path.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=fields)
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, "") for k in fields})
    return path


def write_json(obj, filename: str, results_dir: pathlib.Path | None = None) -> pathlib.Path:
    out_dir = results_dir or results_dir_default()
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / filename
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def read_rows(filename: str = "runs.csv", results_dir: pathlib.Path | None = None) -> list[dict]:
    path = (results_dir or results_dir_default()) / filename
    if not path.exists():
        return []
    with path.open(encoding="utf-8", newline="") as fh:
        return list(csv.DictReader(fh))


def markdown_table(rows: Iterable[dict], columns: list[str] | None = None) -> str:
    """Paste-ready Markdown for REPORT.md."""
    rows = list(rows)
    if not rows:
        return "_(no rows)_"
    cols = columns or list(rows[0])
    head = "| " + " | ".join(cols) + " |"
    rule = "|" + "|".join("---" for _ in cols) + "|"
    body = ["| " + " | ".join(str(r.get(c, "")) for c in cols) + " |" for r in rows]
    return "\n".join([head, rule, *body])


def hbar(values: dict[str, float], width: int = 34, vmax: float | None = None,
         fmt: str = "{:.3f}") -> str:
    """A text bar chart, because a three-way score comparison read as a column of
    floats is where students stop noticing that regression collapsed.

    Text and not matplotlib on purpose: it survives being pasted into REPORT.md, a
    terminal, a PR comment or a grader's diff, and it adds no dependency to a notebook
    whose install cell is already the second-longest step in the lab.
    """
    if not values:
        return "_(no data)_"
    top = vmax if vmax is not None else max(max(values.values()), 1e-9)
    pad = max(len(k) for k in values)
    lines = []
    for k, v in values.items():
        filled = 0 if top <= 0 else max(0, min(width, round(width * v / top)))
        lines.append(f"{k:<{pad}}  {'█' * filled}{'·' * (width - filled)}  "
                     + fmt.format(v))
    return "\n".join(lines)

In [ ]:
%%writefile labkit/replay.py
"""Replay corpus for deck §14.3 — the anti-forgetting mix.

**Why this file exists.** The lab's own measured result on the shipped corpus:

    (b) base + optimized prompt   target 0.495   regression 0.644
    (c) LoRA fine-tune            target 0.990   regression 0.067   <- FAILED

The fine-tune learned the task almost perfectly and lost nearly all of its general
capability — it answers "what is the capital of Vietnam?" with a triage JSON object.
The cause is not LoRA: it is a training corpus in which *every* input is a ticket and
*every* answer is JSON, so "input" and "emit triage JSON" become the same thing. Deck
§14.3 names the remedy — mix 1-5% general data back in — and nothing in the lab tested
it. This is that data.

**Why hand-written and in-repo rather than a HF dataset.** Three reasons, in order of
how much they matter:

1. **Decontamination is checkable.** The regression group is scored on 15 specific
   questions. A downloaded general-instruction set may or may not contain them; a
   file you can read cannot hide them, and `data.assert_replay_decontaminated()`
   proves it with a number.
2. **Shape has to match.** Replay examples must render as a plain user turn, because
   that is how the regression probe asks. A dataset with its own system prompts would
   introduce a second prompt shape and confound the one variable under test.
3. **It runs offline.** A Kaggle session with internet disabled still trains.

**Answers are deliberately short and plain.** They are here to remind the model that
"answer the question in Vietnamese prose" remains a legal behaviour — not to teach it
new facts. A long answer would spend supervised tokens teaching style, which is a
second variable.

Every item is checked against `data/eval_regression.jsonl` before use. Adding items is
encouraged; the assert is what keeps that safe.
"""
from __future__ import annotations

# 47 items. Kept disjoint from the 15 regression-eval questions by topic, and verified
# by Jaccard overlap at import time in the notebook — not by eyeballing.
REPLAY: list[dict] = [
    # --- general knowledge (geography / history / civics) ---
    {"instruction": "Nước nào có diện tích lớn nhất thế giới?", "input": "",
     "output": "Liên bang Nga là quốc gia có diện tích lớn nhất thế giới, khoảng 17 triệu km²."},
    {"instruction": "Đỉnh núi cao nhất Việt Nam tên là gì?", "input": "",
     "output": "Đó là đỉnh Fansipan ở tỉnh Lào Cai, cao 3.147 m so với mực nước biển."},
    {"instruction": "Việt Nam có bao nhiêu tỉnh và thành phố trực thuộc trung ương?", "input": "",
     "output": "Việt Nam có 63 đơn vị hành chính cấp tỉnh, gồm 58 tỉnh và 5 thành phố trực thuộc trung ương."},
    {"instruction": "Đồng tiền chính thức của Nhật Bản gọi là gì?", "input": "",
     "output": "Đồng tiền chính thức của Nhật Bản là yên (JPY)."},
    {"instruction": "Ai được coi là người phát minh ra bóng đèn sợi đốt thương mại?", "input": "",
     "output": "Thomas Edison thường được ghi công là người thương mại hoá bóng đèn sợi đốt vào năm 1879."},
    {"instruction": "Hệ Mặt Trời có bao nhiêu hành tinh?", "input": "",
     "output": "Hệ Mặt Trời có 8 hành tinh, từ Sao Thuỷ đến Sao Hải Vương."},
    {"instruction": "Vạn Lý Trường Thành nằm ở quốc gia nào?", "input": "",
     "output": "Vạn Lý Trường Thành nằm ở Trung Quốc."},
    {"instruction": "Lá cờ Việt Nam có những màu gì?", "input": "",
     "output": "Cờ Việt Nam có nền đỏ và một ngôi sao vàng năm cánh ở giữa."},

    # --- arithmetic / simple quantitative reasoning ---
    {"instruction": "Tính 15 nhân 12.", "input": "",
     "output": "15 × 12 = 180."},
    {"instruction": "Một hình vuông có cạnh 7 cm. Diện tích bằng bao nhiêu?", "input": "",
     "output": "Diện tích = 7 × 7 = 49 cm²."},
    {"instruction": "30% của 250 là bao nhiêu?", "input": "",
     "output": "30% của 250 là 75."},
    {"instruction": "Nếu một món hàng giá 400 nghìn đồng được giảm 25%, giá sau giảm là bao nhiêu?", "input": "",
     "output": "Giảm 25% của 400 nghìn là 100 nghìn, nên giá sau giảm là 300 nghìn đồng."},
    {"instruction": "Số nguyên tố nhỏ nhất lớn hơn 20 là số nào?", "input": "",
     "output": "Đó là 23."},
    {"instruction": "Trung bình của 4, 8 và 15 là bao nhiêu?", "input": "",
     "output": "(4 + 8 + 15) / 3 = 27 / 3 = 9."},

    # --- translation (both directions, none matching the eval item) ---
    {"instruction": "Dịch câu sau sang tiếng Anh: 'Hôm nay trời mưa rất to'.", "input": "",
     "output": "It is raining very heavily today."},
    {"instruction": "Dịch sang tiếng Việt: 'Please send me the invoice tomorrow'.", "input": "",
     "output": "Vui lòng gửi cho tôi hoá đơn vào ngày mai."},
    {"instruction": "Dịch sang tiếng Anh: 'Cảm ơn bạn đã giúp đỡ tôi'.", "input": "",
     "output": "Thank you for helping me."},

    # --- writing / rewriting ---
    {"instruction": "Viết một câu chào mở đầu email cho khách hàng doanh nghiệp.", "input": "",
     "output": "Kính gửi Anh/Chị, cảm ơn Anh/Chị đã quan tâm tới sản phẩm của chúng tôi."},
    {"instruction": "Viết lại câu sau cho lịch sự hơn: 'Gửi báo giá ngay'.", "input": "",
     "output": "Anh/Chị vui lòng gửi giúp em báo giá khi thuận tiện nhé, em xin cảm ơn."},
    {"instruction": "Đặt một tiêu đề ngắn cho bài viết về lợi ích của việc đi bộ mỗi ngày.", "input": "",
     "output": "Mỗi ngày ba mươi phút đi bộ: món đầu tư rẻ nhất cho sức bền."},
    {"instruction": "Viết một câu cảm ơn đồng nghiệp đã hỗ trợ dự án.", "input": "",
     "output": "Cảm ơn bạn đã hỗ trợ nhiệt tình, dự án về đích đúng hạn là nhờ phần việc của bạn."},
    {"instruction": "Tóm tắt trong một câu: 'Công ty mở thêm hai chi nhánh tại Đà Nẵng và Cần Thơ trong quý này, đồng thời tuyển thêm 40 nhân viên.'", "input": "",
     "output": "Trong quý này công ty mở hai chi nhánh mới ở Đà Nẵng và Cần Thơ và tuyển thêm 40 nhân viên."},

    # --- explanation / definition ---
    # Worded to stay off the eval set's "Giải thích ngắn gọn quang hợp là gì." shape:
    # that phrasing scores Jaccard 0.60 against it, right on the decontamination limit.
    {"instruction": "Lạm phát ảnh hưởng thế nào tới tiền tiết kiệm của một gia đình?", "input": "",
     "output": "Lạm phát là hiện tượng mức giá chung của hàng hoá và dịch vụ tăng lên theo thời gian, làm giảm sức mua của đồng tiền."},
    {"instruction": "Mã bưu chính dùng để làm gì?", "input": "",
     "output": "Mã bưu chính giúp phân loại và định tuyến thư từ, bưu kiện tới đúng khu vực giao nhận."},
    {"instruction": "Sự khác nhau giữa RAM và ổ cứng là gì?", "input": "",
     "output": "RAM là bộ nhớ tạm, rất nhanh nhưng mất dữ liệu khi tắt máy; ổ cứng lưu dữ liệu lâu dài nhưng chậm hơn nhiều."},
    {"instruction": "HTTPS khác HTTP ở điểm nào?", "input": "",
     "output": "HTTPS mã hoá dữ liệu trên đường truyền bằng TLS, còn HTTP gửi dữ liệu dạng rõ nên có thể bị đọc hoặc sửa."},
    {"instruction": "Vì sao nên sao lưu dữ liệu định kỳ?", "input": "",
     "output": "Vì ổ đĩa có thể hỏng, thiết bị có thể mất và người dùng có thể xoá nhầm; bản sao lưu là cách duy nhất phục hồi lại được."},
    {"instruction": "Giải thích ngắn gọn thế nào là năng lượng tái tạo.", "input": "",
     "output": "Năng lượng tái tạo là năng lượng lấy từ những nguồn tự bù đắp trong thời gian ngắn như nắng, gió, nước và sinh khối."},
    {"instruction": "Vì sao cần uống đủ nước mỗi ngày?", "input": "",
     "output": "Nước tham gia vào điều hoà nhiệt độ, vận chuyển dưỡng chất và đào thải chất cặn, nên thiếu nước làm cơ thể hoạt động kém đi."},

    # --- lists ---
    {"instruction": "Kể tên ba loại rau ăn lá phổ biến ở Việt Nam.", "input": "",
     "output": "Rau muống, rau cải xanh và mồng tơi."},
    {"instruction": "Kể tên hai ngôn ngữ lập trình thường dùng cho khoa học dữ liệu.", "input": "",
     "output": "Python và R."},
    {"instruction": "Nêu hai cách tiết kiệm điện trong gia đình.", "input": "",
     "output": "Tắt thiết bị khi không dùng và thay đèn sợi đốt bằng đèn LED."},
    {"instruction": "Kể tên ba phương tiện giao thông công cộng.", "input": "",
     "output": "Xe buýt, tàu điện và tàu hoả."},

    # --- light reasoning ---
    {"instruction": "Nếu hôm nay là thứ Tư thì ba ngày sau là thứ mấy?", "input": "",
     "output": "Ba ngày sau thứ Tư là thứ Bảy."},
    {"instruction": "Một xe chạy 60 km trong 45 phút. Tốc độ trung bình là bao nhiêu km/h?", "input": "",
     "output": "45 phút là 0,75 giờ, nên tốc độ trung bình là 60 / 0,75 = 80 km/h."},
    {"instruction": "Sắp xếp các số sau theo thứ tự tăng dần: 12, 3, 47, 9.", "input": "",
     "output": "3, 9, 12, 47."},
    {"instruction": "Trong câu 'Con mèo đuổi con chuột', ai là chủ ngữ?", "input": "",
     "output": "Chủ ngữ là 'con mèo'."},

    # --- everyday how-to ---
    {"instruction": "Làm thế nào để pha một ly trà xanh đúng cách?", "input": "",
     "output": "Tráng ấm bằng nước nóng, dùng nước khoảng 80 độ C, hãm lá trà chừng hai phút rồi rót ra ngay để trà không bị đắng."},
    {"instruction": "Nên làm gì đầu tiên khi điện thoại bị rơi xuống nước?", "input": "",
     "output": "Tắt máy ngay, lau khô bên ngoài và để khô tự nhiên ở nơi thoáng; không sấy nóng và không cắm sạc."},
    {"instruction": "Cách bảo quản rau xanh trong tủ lạnh cho lâu hơn?", "input": "",
     "output": "Để rau thật khô, bọc bằng giấy hoặc khăn thấm rồi cho vào hộp kín và đặt ở ngăn mát, tránh để cạnh trái cây chín."},
    {"instruction": "Làm sao đặt một mật khẩu mạnh mà vẫn nhớ được?", "input": "",
     "output": "Ghép bốn từ ngẫu nhiên không liên quan thành một cụm dài, thêm một chữ số và một ký tự đặc biệt; độ dài quan trọng hơn sự phức tạp."},

    # --- opinion / short advice, keeps discursive answering alive ---
    {"instruction": "Một lời khuyên cho người mới bắt đầu học lập trình?", "input": "",
     "output": "Viết chương trình nhỏ mỗi ngày và đọc lại code của chính mình sau một tuần — thói quen đó dạy nhiều hơn là đọc thêm một quyển sách."},
    {"instruction": "Nên chuẩn bị gì trước một buổi phỏng vấn xin việc?", "input": "",
     "output": "Đọc kỹ mô tả công việc, chuẩn bị hai ví dụ cụ thể về việc mình đã làm, và soạn sẵn vài câu hỏi cho nhà tuyển dụng."},
    {"instruction": "Vì sao nên đọc điều khoản trước khi cài một ứng dụng lạ?", "input": "",
     "output": "Vì đó là nơi ghi ứng dụng thu thập dữ liệu gì và chia sẻ với ai; quyền truy cập danh bạ hay vị trí thường được xin ở đó."},
    {"instruction": "Làm việc nhóm hiệu quả cần điều gì nhất?", "input": "",
     "output": "Sự rõ ràng: ai làm việc gì, xong khi nào, và thông tin gì cần chia sẻ cho ai."},

    # --- format-adjacent but NOT triage: proves the model can still choose a format ---
    {"instruction": "Liệt kê ba bước gửi hàng qua bưu điện, mỗi bước một dòng.", "input": "",
     "output": "1. Đóng gói và ghi rõ địa chỉ người nhận.\n2. Mang tới bưu cục và chọn dịch vụ chuyển phát.\n3. Giữ biên nhận để tra mã vận đơn."},
    {"instruction": "Viết một câu mô tả thời tiết Hà Nội vào tháng Ba.", "input": "",
     "output": "Tháng Ba Hà Nội thường ẩm và nhiều sương, nhiệt độ dịu, đôi khi có mưa phùn kéo dài."},
]


def load(extra: list[dict] | None = None) -> list[dict]:
    """The replay pool, optionally extended. Always run it through
    `data.assert_replay_decontaminated()` against the regression eval set first."""
    pool = [dict(r) for r in REPLAY]
    if extra:
        pool += [dict(r) for r in extra]
    for r in pool:
        r.setdefault("input", "")
        if not r.get("instruction") or not r.get("output"):
            raise ValueError(f"replay record needs instruction+output: {r!r}")
    return pool

In [ ]:
%%writefile labkit/data.py
"""Chat templating, loss masking, and dataset prep.

The deck's claim (§13.2) is that loss masking and the chat template decide more
outcomes than every LoRA variant combined. This module exists so you can *see* the
mask rather than trust a library flag.

The masking technique used here renders the conversation to **text** around each
assistant turn, then maps character spans onto tokens:

    prefix = apply_chat_template(messages[:i], add_generation_prompt=True)   # text
    upto   = apply_chat_template(messages[:i+1])                             # text
    supervised characters = [len(prefix), len(upto))
    supervised tokens     = tokens whose offsets fall inside that range

**Why not just diff the token lists?** Because that is wrong on real templates, and
Qwen3.5 is the counter-example. Its prefix ends `<think>\n` while the full render
continues `\n</think>`; tokenized separately the trailing `\n` is its own token, but
tokenized together `\n\n` merges into a *different single token*. The token lists are
therefore not prefix-related even though the strings are. Diffing tokens raises a
false alarm at best and silently mis-masks at worst.

Characters do not have this problem, and `return_offsets_mapping` gives the mapping
back to tokens — including for special tokens like `<|im_end|>`, which must stay
supervised because it is the model's stop signal.
"""
from __future__ import annotations

import statistics
import warnings

from .config import NAIVE_PROMPT
from dataclasses import dataclass, field

IGNORE_INDEX = -100

# What the loss is computed over. Deck §13.5: on a reasoning base, this choice can
# preserve or destroy the model's reasoning behaviour, and the safe option is
# model-dependent — so it is a parameter, not a constant.
MASK_MODES = ("assistant-only", "masked-think", "response-only", "everything")


@dataclass
class Example:
    input_ids: list[int]
    labels: list[int]
    n_supervised: int
    n_total: int

    @property
    def supervised_fraction(self) -> float:
        return self.n_supervised / max(1, self.n_total)


class TemplateNotPrefixStable(RuntimeError):
    """Raised when apply_chat_template(msgs[:i+1]) does not start with the i-prefix.

    If you hit this, the template is rewriting earlier turns as the conversation grows
    (some templates move a system prompt, or strip reasoning from previous turns).
    Masking by difference is unsafe there — inspect the rendered strings before
    training, and see `thinking_survives()`.
    """


def _render(tokenizer, messages, add_generation_prompt: bool, enable_thinking: bool | None) -> str:
    """Render a conversation to text (never to tokens — see the module docstring)."""
    kwargs = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    if enable_thinking is not None:
        # Not every template accepts this; pass it only when asked for.
        kwargs["enable_thinking"] = enable_thinking
    out = tokenizer.apply_chat_template(messages, **kwargs)
    if isinstance(out, list):
        out = out[0]
    return out


def _encode_with_offsets(tokenizer, text: str):
    """Tokenize rendered chat text, returning (ids, char_offsets).

    `add_special_tokens=False` because the template already emits them; this was
    verified to reproduce `apply_chat_template(tokenize=True)` exactly on Qwen3.5.
    """
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc.get("offset_mapping")
    if offsets is None:
        raise RuntimeError(
            "This tokenizer did not return offset mappings (is it a 'slow' tokenizer?). "
            "Loss masking here needs a fast tokenizer: "
            "AutoTokenizer.from_pretrained(..., use_fast=True)."
        )
    return list(enc["input_ids"]), list(offsets)


def find_span(haystack: list[int], needle: list[int], start: int = 0) -> int:
    """Index of `needle` in `haystack` at/after `start`, or -1."""
    if not needle:
        return -1
    n = len(needle)
    for i in range(start, len(haystack) - n + 1):
        if haystack[i:i + n] == needle:
            return i
    return -1


def build_example(
    tokenizer,
    messages: list[dict],
    max_length: int = 1024,
    mask_mode: str = "assistant-only",
    enable_thinking: bool | None = None,
    think_open: str = "<think>",
    think_close: str = "</think>",
) -> Example:
    """Tokenize a conversation and build the label mask.

    mask_mode:
      "assistant-only" — supervise every assistant token (the usual SFT default)
      "masked-think"   — supervise the assistant turn but NOT its reasoning block
      "response-only"  — supervise only what follows </think> (strictest)
      "everything"     — supervise all tokens, prompt included. This is the classic
                         bug (§16: "model writes your question back at you"); it is
                         selectable so NB1 can show you what it looks like.
    """
    if mask_mode not in MASK_MODES:
        raise ValueError(f"mask_mode must be one of {MASK_MODES}, got {mask_mode!r}")

    full_text = _render(tokenizer, messages, False, enable_thinking)
    full_ids, offsets = _encode_with_offsets(tokenizer, full_text)
    labels = [IGNORE_INDEX] * len(full_ids)

    if mask_mode == "everything":
        labels = list(full_ids)
    else:
        for i, msg in enumerate(messages):
            if msg.get("role") != "assistant":
                continue
            prefix_text = _render(tokenizer, messages[:i], True, enable_thinking)
            upto_text = _render(tokenizer, messages[:i + 1], False, enable_thinking)
            if not full_text.startswith(prefix_text):
                raise TemplateNotPrefixStable(
                    f"turn {i}: rendering messages[:{i}] with add_generation_prompt is not "
                    "a text prefix of the full render. This template rewrites earlier "
                    "turns as the conversation grows; masking by difference is unsafe."
                )
            start_char, end_char = len(prefix_text), len(upto_text)
            if mask_mode in ("masked-think", "response-only"):
                start_char = _skip_reasoning_chars(
                    full_text, start_char, end_char, think_close)
            for j, (a, b) in enumerate(offsets):
                if b > a and a >= start_char and b <= end_char:
                    labels[j] = full_ids[j]

    if len(full_ids) > max_length:
        full_ids = full_ids[:max_length]
        labels = labels[:max_length]

    return Example(
        input_ids=full_ids,
        labels=labels,
        n_supervised=sum(1 for x in labels if x != IGNORE_INDEX),
        n_total=len(full_ids),
    )


def _skip_reasoning_chars(text: str, start: int, end: int, think_close: str) -> int:
    """Move `start` past a closing reasoning tag inside text[start:end), if present.

    Character-based for the same reason as the main path: tag boundaries do not align
    with token boundaries.

    **When this actually fires.** Qwen3.5 does emit `<think>\n\n</think>\n\n` for a
    non-reasoning answer — but it emits it as part of the *generation prompt*, i.e.
    inside `prefix_text`. So `start` is already past `</think>` and there is nothing
    left in `[start, end)` to skip: on a corpus of plain answers `masked-think` and
    `response-only` produce a mask identical to `assistant-only`. This only does work
    when the assistant *content* carries its own reasoning block, which is what a
    §13.5 experiment needs its training data to look like.
    """
    at = text.find(think_close, start, end)
    if at == -1:
        return start           # no reasoning block in this turn — nothing to skip
    return at + len(think_close)


# --- inspection helpers: the point of NB1 -----------------------------------

def decode_supervised(tokenizer, ex: Example) -> str:
    """Exactly the text the loss is computed on. Read this before you train."""
    kept = [t for t, l in zip(ex.input_ids, ex.labels) if l != IGNORE_INDEX]
    return tokenizer.decode(kept, skip_special_tokens=False)


def decode_masked(tokenizer, ex: Example) -> str:
    """The complement: everything the model sees but is not scored on."""
    dropped = [t for t, l in zip(ex.input_ids, ex.labels) if l == IGNORE_INDEX]
    return tokenizer.decode(dropped, skip_special_tokens=False)


def thinking_survives(tokenizer, think_open="<think>", think_close="</think>") -> dict:
    """Deck §16: some chat templates DELETE reasoning blocks inside apply_chat_template.

    If that happens, the reasoning traces in your dataset never reach the loss — and
    nothing errors. Run this once per base model, before training.
    """
    body = "buoc 1: kiem tra. buoc 2: tra loi."
    messages = [
        {"role": "user", "content": "2+2?"},
        {"role": "assistant", "content": f"{think_open}{body}{think_close}4"},
    ]
    try:
        rendered = tokenizer.apply_chat_template(messages, tokenize=False)
    except Exception as exc:                      # pragma: no cover - template variance
        return {
            "ok": False,
            "error": repr(exc),
            "rendered": None,
            "open_tag_present": False,
            "body_present": False,
            "verdict": f"TEMPLATE FAILED TO RENDER: {exc}",
        }
    if isinstance(rendered, list):
        rendered = rendered[0]
    return {
        "ok": body in rendered,
        "open_tag_present": think_open in rendered,
        "body_present": body in rendered,
        "rendered": rendered,
        "verdict": (
            "reasoning preserved — safe to train on traces"
            if body in rendered
            else "TEMPLATE STRIPS REASONING — your traces will never reach the loss"
        ),
    }


def token_stats(lengths: list[int]) -> dict:
    """p50/p95/p99 so `max_length` is a measurement, not a guess (deck §13)."""
    if not lengths:
        return {"n": 0}
    ordered = sorted(lengths)

    def pct(p: float) -> int:
        # Nearest-rank percentile: index = ceil(p * n) - 1. Chosen over
        # round(p*(n-1)) because Python's round() is banker's rounding, which makes
        # the median of 1..100 land on 51 instead of 50 — a silent off-by-one in the
        # number students use to set `max_length`.
        import math
        idx = min(len(ordered) - 1, max(0, math.ceil(p * len(ordered)) - 1))
        return ordered[idx]

    return {
        "n": len(ordered),
        "mean": round(statistics.fmean(ordered), 1),
        "p50": pct(0.50),
        "p95": pct(0.95),
        "p99": pct(0.99),
        "max": ordered[-1],
        "suggested_max_length": _round_pow2(pct(0.95)),
    }


def _round_pow2(n: int) -> int:
    p = 256
    while p < n and p < 32768:
        p *= 2
    return p


def to_messages(record: dict, system: str | None = NAIVE_PROMPT) -> list[dict]:
    """Normalize the common instruction formats to a messages list.

    **`system` decides what the model is trained to condition on, and it has to be the
    same thing evaluation will hand it.** This is F-31, the defect that made every
    adapter in the lab score 0.000:

        old (system=None)   user: "<full schema + enum list>\n\n<ticket>"  -> JSON
        eval always sent    system: "Phân loại ticket sau."  +  user: "<ticket>"

    Training folded the whole instruction -- keys, enum values, "chỉ trả về JSON" -- into
    the user turn, and there was no system message at all. Evaluation then asked for the
    same task from a prompt containing none of it, in a role structure the model had
    never seen. NB5's comment ("the behaviour moved into the weights, so the prompt can
    shrink") is the right ambition, but the prompt was shrunk at evaluation time without
    ever training the shrunk form. The model answered the question it was actually
    asked -- vague instruction, no schema -- with fluent Vietnamese prose, and scored
    zero on both target and format while its training loss looked perfect.

    So the default is now the *evaluation* prompt: the ticket alone in the user turn,
    `NAIVE_PROMPT` as the system message. What the model has to internalise is the label
    space, which is exactly the thing fine-tuning is supposed to buy you here.

    Pass `system=None` to reproduce the old instruction-in-the-user-turn shape -- NB1
    uses it to show the two renders side by side.
    """
    if "messages" in record and isinstance(record["messages"], list):
        return record["messages"]
    instruction = record.get("instruction") or record.get("prompt") or record.get("question") or ""
    context = record.get("input") or record.get("context") or ""
    output = record.get("output") or record.get("response") or record.get("answer") or ""
    if system is not None and context:
        return [
            {"role": "system", "content": system},
            {"role": "user", "content": str(context).strip()},
            {"role": "assistant", "content": str(output).strip()},
        ]
    user = f"{instruction}\n\n{context}".strip() if context else str(instruction).strip()
    return [
        {"role": "user", "content": user},
        {"role": "assistant", "content": str(output).strip()},
    ]


def to_training_dataset(
    tokenizer,
    records: list[dict],
    max_length: int = 1024,
    mask_mode: str = "assistant-only",
    enable_thinking: bool | None = None,
    think_open: str = "<think>",
) -> list[dict]:
    """Pre-tokenize records into {input_ids, labels} using the mask NB1 verified.

    **Why pre-tokenize instead of letting TRL do it.** TRL's `assistant_only_loss`
    derives its mask from `{% generation %}` markers in the chat template. Qwen3.5's
    template has none, so that flag yields a mask of ZERO supervised tokens — and
    transformers only *warns*. Training runs to completion and the numbers are
    meaningless. Run `scripts/check_mask_agreement.py` to see it on your own base model.

    Training on the output of `build_example()` means the loss covers exactly the tokens
    the student decoded and asserted on in NB1. No flag in between to be wrong.

    Trade-off: pre-tokenized labels cannot be `packing`-ed (packing concatenates
    examples and would invalidate the label alignment). Correctness of the mask outranks
    the throughput here.
    """
    # `masked-think` and `response-only` differ from `assistant-only` only when the
    # assistant CONTENT carries its own reasoning block. Qwen3.5's generation prompt
    # already closes an empty `<think></think>`, so on a corpus of plain answers the
    # skip has nothing left to skip and all three modes emit an identical mask — see
    # `_skip_reasoning_chars`. The shipped triage corpus is exactly that: 250 bare-JSON
    # answers. A student who sets MASK_MODE for the §13.5 contrast would otherwise see
    # no difference and no reason why, so say it out loud instead of being silently inert.
    if mask_mode in ("masked-think", "response-only"):
        if not any(think_open in (r.get("output") or "") for r in records):
            warnings.warn(
                f"mask_mode={mask_mode!r} is a no-op on this corpus: none of the "
                f"{len(records)} records carry a {think_open} block in their answer, "
                "and the chat template closes its empty reasoning block inside the "
                "generation prompt. The resulting mask is identical to 'assistant-only'. "
                "Exercising deck §13.5 needs training answers that contain real traces "
                "— see 'Đổi dataset của riêng bạn' in README.md.",
                RuntimeWarning,
                stacklevel=2,
            )

    out: list[dict] = []
    for r in records:
        ex = build_example(
            tokenizer, to_messages(r), max_length=max_length,
            mask_mode=mask_mode, enable_thinking=enable_thinking,
        )
        if ex.n_supervised == 0:
            continue                       # nothing to learn from; drop it
        out.append({"input_ids": ex.input_ids, "labels": ex.labels,
                    "attention_mask": [1] * len(ex.input_ids)})
    if not out:
        raise RuntimeError(
            "every example ended up with zero supervised tokens — the mask is wrong. "
            "Re-run NB1 and read results/mask_proof.json before training."
        )
    return out


def split(records: list, train_frac: float = 0.9, seed: int = 42) -> tuple[list, list]:
    """Deterministic split. Same seed in every notebook so runs stay comparable."""
    import random
    rng = random.Random(seed)
    idx = list(range(len(records)))
    rng.shuffle(idx)
    cut = int(len(idx) * train_frac)
    return [records[i] for i in idx[:cut]], [records[i] for i in idx[cut:]]


def prompt_alignment(tokenizer, record: dict, system: str | None = NAIVE_PROMPT,
                     enable_thinking: bool | None = False) -> dict:
    """Does the prompt we TRAIN on match the prompt evaluation will send?

    F-31: it did not, and nothing checked. Training folded the task instruction into the
    user turn while evaluation sent a short system prompt and a bare input, so the
    fine-tune was asked to continue a prompt it had never seen. Training loss looked
    perfect and every adapter scored 0.000 on the task.

    A loss mask can be right while the thing being conditioned on is wrong. NB1 proves
    the mask; this proves the prompt. Returns the two renders and whether the training
    text starts with the evaluation text -- if it does not, the fine-tune cannot transfer
    no matter how well it trains.
    """
    msgs = to_messages(record, system=system)
    train_text = _render(tokenizer, msgs, False, enable_thinking)

    ask = [m for m in msgs if m["role"] != "assistant"]
    eval_text = _render(tokenizer, ask, True, enable_thinking)

    return {
        "train_render": train_text,
        "eval_render": eval_text,
        "eval_prompt_is_prefix_of_training": train_text.startswith(eval_text),
        "supervised_tail": train_text[len(eval_text):] if train_text.startswith(eval_text) else None,
    }


# --- Kaggle additions --------------------------------------------------------------

def assert_prompt_alignment(tokenizer, records: list[dict],
                            system: str | None = NAIVE_PROMPT,
                            enable_thinking: bool | None = False,
                            sample: int = 5) -> dict:
    """Fail the run when training and evaluation do not render the same scaffold.

    `prompt_alignment()` in the repo returns a dict and nothing reads it. F-31 is the
    reason that matters: training folded the whole instruction into the user turn while
    evaluation sent a short system prompt and a bare ticket, so every adapter was asked
    to continue a prompt it had never seen. Training loss looked perfect; the task score
    was 0.000 for four runs across two sessions.

    A check whose failure mode is "a dict nobody printed" is not a check. This raises.
    """
    checked = []
    for rec in records[:max(1, sample)]:
        info = prompt_alignment(tokenizer, rec, system=system,
                               enable_thinking=enable_thinking)
        checked.append(info)
        if not info["eval_prompt_is_prefix_of_training"]:
            raise RuntimeError(
                "TRAIN/EVAL PROMPT MISMATCH (F-31). The text evaluation sends is not a "
                "prefix of the text training supervises, so the fine-tune will be asked "
                "to continue a prompt it never saw.\n"
                f"--- eval render ---\n{info['eval_render']!r}\n"
                f"--- train render ---\n{info['train_render'][:400]!r}"
            )
    tails = [len(c["supervised_tail"] or "") for c in checked]
    return {
        "n_checked": len(checked),
        "all_aligned": True,
        "eval_render_example": checked[0]["eval_render"],
        "supervised_tail_example": checked[0]["supervised_tail"],
        "supervised_tail_chars": {"min": min(tails), "max": max(tails)},
    }


def _word_set(text: str) -> set:
    import re as _re
    from .evaluate import normalize
    return set(_re.findall(r"\w+", normalize(text)))


def assert_replay_decontaminated(replay: list[dict], eval_regression: list[dict],
                                 jaccard_max: float = 0.6) -> dict:
    """Replay data must not be the regression eval set with the answers filled in.

    This is the failure mode that would make the §14.3 fix look like it worked when it
    did not: mix the eval questions into training, watch `regression` go back up, and
    report a win that is memorisation. Two checks, both deliberately blunt:

      * no normalized-exact-match instruction, and
      * no pair above `jaccard_max` word overlap (catches "Thủ đô Việt Nam là gì?" vs
        "Thủ đô của Việt Nam là thành phố nào?", which exact match would miss).

    Returns the worst overlap it found, so a report can quote a number rather than the
    word "decontaminated".
    """
    eval_norm = {}
    from .evaluate import normalize
    for r in eval_regression:
        eval_norm[normalize(r.get("instruction", ""))] = r.get("instruction", "")

    worst = {"jaccard": 0.0, "replay": None, "eval": None}
    exact: list[tuple[str, str]] = []
    for rec in replay:
        q = rec.get("instruction", "")
        if normalize(q) in eval_norm:
            exact.append((q, eval_norm[normalize(q)]))
        qs = _word_set(q)
        for enorm, eraw in eval_norm.items():
            es = _word_set(eraw)
            if not qs or not es:
                continue
            j = len(qs & es) / len(qs | es)
            if j > worst["jaccard"]:
                worst = {"jaccard": round(j, 3), "replay": q, "eval": eraw}

    if exact:
        raise RuntimeError(
            f"replay corpus CONTAMINATED: {len(exact)} instruction(s) are also in "
            f"eval_regression.jsonl, e.g. {exact[0][0]!r}. Scoring the regression group "
            "on questions the model was trained on measures memorisation, not retained "
            "capability."
        )
    if worst["jaccard"] > jaccard_max:
        raise RuntimeError(
            f"replay corpus NEAR-DUPLICATE of an eval question "
            f"(Jaccard {worst['jaccard']:.2f} > {jaccard_max}):\n"
            f"  replay: {worst['replay']!r}\n  eval:   {worst['eval']!r}"
        )
    return {"n_replay": len(replay), "n_eval_regression": len(eval_regression),
            "exact_collisions": 0, "worst_jaccard": worst["jaccard"],
            "worst_pair": [worst["replay"], worst["eval"]],
            "jaccard_threshold": jaccard_max}


def mix_replay(triage: list[dict], replay: list[dict], fraction: float,
               seed: int = 42) -> tuple[list[dict], dict]:
    """Fold `fraction` of replay data into the triage corpus (deck §14.3).

    `fraction` is of the MIXED total, not of the triage count — "5% replay" should mean
    5% of what the optimizer sees. So with 225 tickets and fraction=0.05 the mix carries
    12 replay examples (12 / 237 = 5.1%), not 11.25 (which would be 4.8% of the total).

    Replay examples are sampled without replacement while the pool lasts and then cycle,
    deterministically under `seed`, and the mix is shuffled so replay is not all in the
    last steps — a tail-loaded mix trains the anti-forgetting signal only at the end of
    the cosine schedule, when the learning rate is nearly zero.

    Returns (mixed_records, manifest). The manifest goes into results/ because "we added
    replay data" is not a reproducible statement and "12 of 237 examples, seed 42" is.
    """
    import random
    if not 0.0 <= fraction < 0.5:
        raise ValueError(f"fraction must be in [0, 0.5), got {fraction}")
    if fraction == 0.0 or not replay:
        return list(triage), {"n_triage": len(triage), "n_replay_used": 0,
                              "fraction_requested": fraction,
                              "fraction_achieved": 0.0, "seed": seed}

    n_t = len(triage)
    # n_r / (n_t + n_r) == fraction  =>  n_r = fraction * n_t / (1 - fraction)
    n_r = max(1, round(fraction * n_t / (1.0 - fraction)))

    rng = random.Random(seed)
    pool = list(replay)
    rng.shuffle(pool)
    picked = [pool[i % len(pool)] for i in range(n_r)]

    for rec in picked:
        if (rec.get("input") or "").strip():
            raise ValueError(
                "replay records must carry an empty `input`: the triage examples put the "
                "ticket in `input` (with a system prompt), and replay must render as a "
                "plain user question, or the mix teaches two prompt shapes at once."
            )

    mixed = list(triage) + picked
    rng.shuffle(mixed)
    return mixed, {
        "n_triage": n_t,
        "n_replay_used": n_r,
        "n_replay_pool": len(replay),
        "fraction_requested": fraction,
        "fraction_achieved": round(n_r / (n_t + n_r), 4),
        "seed": seed,
        "replay_instructions": [r.get("instruction", "")[:60] for r in picked],
    }

In [ ]:
%%writefile labkit/evaluate.py
"""Four-group evaluation and the regression gate.

Deck §17: perplexity alone is not evidence. A fine-tune is only a win if it beats a
*well-prompted base model* on the target task **without** quietly losing general
capability. So every run is scored on four groups:

    1. target      — does it do the job you trained it for?
    2. regression  — did it forget everything else? (deck §14.3)
    3. format      — does it obey the output contract?
    4. latency     — what did the win cost at inference time?

and compared against three frozen baselines:

    (a) base + naive prompt
    (b) base + a genuinely optimized prompt   <- the one that matters
    (c) the fine-tune

The graded question is not "did perplexity drop". It is: **did you beat (b), and
would you have noticed if you hadn't.**

Everything here is a pure function over already-generated strings, so the whole
harness is testable on a laptop with no GPU.
"""
from __future__ import annotations

import json
import re
import time
import unicodedata
from dataclasses import dataclass, field, asdict

# --- Vietnamese-aware normalization -----------------------------------------
# Grader footgun this course has hit before: matching an *unaccented* keyword against
# accented Vietnamese output (or vice versa) silently passes or fails everything. Pick
# one normalization and apply it to BOTH sides, always.


def strip_accents(text: str) -> str:
    decomposed = unicodedata.normalize("NFD", text)
    without = "".join(c for c in decomposed if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", without).replace("đ", "d").replace("Đ", "D")


def normalize(text: str, fold_accents: bool = True) -> str:
    text = (text or "").strip().lower()
    text = re.sub(r"\s+", " ", text)
    return strip_accents(text) if fold_accents else text


# --- scorers ----------------------------------------------------------------

def exact_match(pred: str, ref: str, fold_accents: bool = True) -> float:
    return float(normalize(pred, fold_accents) == normalize(ref, fold_accents))


def keyword_recall(pred: str, keywords: list[str], fold_accents: bool = True) -> float:
    """Fraction of required keywords present. BOTH sides normalized identically."""
    if not keywords:
        return 1.0
    hay = normalize(pred, fold_accents)
    hits = sum(1 for k in keywords if normalize(k, fold_accents) in hay)
    return hits / len(keywords)


def _parse_json_loose(pred: str):
    """Best-effort JSON extraction: bare, fenced, or the first {...} block."""
    candidate = (pred or "").strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", candidate, re.S)
    if fence:
        candidate = fence.group(1).strip()
    try:
        return json.loads(candidate)
    except Exception:
        pass
    brace = re.search(r"\{.*\}", candidate, re.S)
    if brace:
        try:
            return json.loads(brace.group(0))
        except Exception:
            return None
    return None


def json_parses(pred: str) -> float:
    """Format compliance: does the output parse as JSON (fenced or bare)?"""
    candidate = pred.strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", candidate, re.S)
    if fence:
        candidate = fence.group(1).strip()
    try:
        json.loads(candidate)
        return 1.0
    except Exception:
        return 0.0


def has_required_keys(pred: str, keys: list[str]) -> float:
    """Fraction of required keys present in the emitted object.

    Uses the SAME loose parser as `triage_field_accuracy`. They diverged at first —
    this one only handled bare/fenced JSON while the target scorer also recovered a
    `{...}` block embedded in prose. A model that answers
    `"Day la ket qua: {...}"` would then score on the target group but zero on format,
    which reads as a formatting failure that did not happen. Two scorers disagreeing
    about whether output *is* JSON makes both numbers untrustworthy.
    """
    obj = _parse_json_loose(pred)
    if not isinstance(obj, dict) or not keys:
        return 0.0
    return sum(1 for k in keys if k in obj) / len(keys)


def valid_reasoning_trace(pred: str, open_tag="<think>", close_tag="</think>") -> float:
    """Deck §13.5: the metric that catches reasoning-trace collapse.

    A trace is valid when the block exists, is closed, and is not empty. Task accuracy
    can rise while this falls to zero — which is the entire point of measuring it.
    """
    if open_tag not in pred or close_tag not in pred:
        return 0.0
    body = pred.split(open_tag, 1)[1].split(close_tag, 1)[0]
    return float(len(body.strip()) >= 10)


TRIAGE_KEYS = ["intent", "urgency", "product", "sentiment"]


def triage_field_accuracy(pred: str, label: dict, keys: list[str] | None = None) -> float:
    """Target-task scorer: fraction of the 4 triage fields predicted correctly.

    Partial credit on purpose — a model that gets intent right but urgency wrong is
    genuinely better than one that gets neither, and an all-or-nothing scorer hides
    the training signal you are trying to observe across runs.

    `product` is compared with accents folded (it is copied from free text); the
    categorical fields are compared exactly after lowercasing, because they come from
    a closed vocabulary and near-misses there are errors, not spelling.
    """
    keys = keys or TRIAGE_KEYS
    obj = _parse_json_loose(pred)
    if not isinstance(obj, dict):
        return 0.0
    hits = 0
    for k in keys:
        got, want = obj.get(k), label.get(k)
        if got is None or want is None:
            continue
        if k == "product":
            hits += int(normalize(str(got)) == normalize(str(want)))
        else:
            hits += int(str(got).strip().lower() == str(want).strip().lower())
    return hits / len(keys)


# --- aggregation ------------------------------------------------------------

@dataclass
class GroupScores:
    target: float = 0.0
    regression: float = 0.0
    format: float = 0.0
    latency_ms: float = 0.0
    n: int = 0
    extra: dict = field(default_factory=dict)

    def as_dict(self) -> dict:
        return asdict(self)


@dataclass
class Verdict:
    passed: bool
    reasons: list[str]
    target_delta: float
    regression_delta: float

    def as_dict(self) -> dict:
        return asdict(self)


# How much general capability you are allowed to trade for target-task gain before the
# run is called a regression. 2 points is tight on purpose: the deck's argument is that
# LoRA's forgetting advantage is real, so a correctly configured run should clear it.
REGRESSION_TOLERANCE = 0.02
MIN_TARGET_GAIN = 0.0


def regression_gate(
    tuned: GroupScores,
    optimized_baseline: GroupScores,
    tolerance: float = REGRESSION_TOLERANCE,
    min_gain: float = MIN_TARGET_GAIN,
) -> Verdict:
    """The gate the lab is graded on.

    Passing requires BOTH:
      * target score strictly better than baseline (b) by at least `min_gain`, and
      * general-capability score not worse than (b) by more than `tolerance`.
    """
    target_delta = tuned.target - optimized_baseline.target
    regression_delta = tuned.regression - optimized_baseline.regression

    # The verdict is derived from the numbers, not from the prose that explains them.
    # This used to be `not any(r.startswith(("target", "general")) for r in reasons)` --
    # correct, but only for as long as nobody reworded a message. A graded pass/fail
    # decided by string prefixes is the same class of quiet failure this lab is about.
    beat_baseline = target_delta > min_gain
    kept_capability = regression_delta >= -tolerance

    reasons: list[str] = []
    if not beat_baseline:
        reasons.append(
            f"target task did not beat the optimized-prompt baseline "
            f"({tuned.target:.3f} vs {optimized_baseline.target:.3f}, delta {target_delta:+.3f}). "
            "A fine-tune that loses to a better prompt is not a fine-tune you should ship."
        )
    if not kept_capability:
        reasons.append(
            f"general capability regressed by {abs(regression_delta):.3f} "
            f"(tolerance {tolerance:.3f}). See deck §14.3 — add 1-5% replay data."
        )
    if not reasons:
        reasons.append(
            f"beat the optimized-prompt baseline by {target_delta:+.3f} with "
            f"{regression_delta:+.3f} general-capability change."
        )
    return Verdict(
        passed=beat_baseline and kept_capability,
        reasons=reasons,
        target_delta=target_delta,
        regression_delta=regression_delta,
    )


def comparison_table(scores: dict[str, GroupScores]) -> list[dict]:
    """Rows for REPORT.md. Key order is the reading order: a, b, then the fine-tune."""
    rows = []
    for name, s in scores.items():
        rows.append({
            "run": name,
            "target": round(s.target, 4),
            "regression": round(s.regression, 4),
            "format": round(s.format, 4),
            "latency_ms": round(s.latency_ms, 1),
            "n": s.n,
        })
    return rows


class Timer:
    """Context manager for the latency group. Wall clock, per generation."""

    def __init__(self) -> None:
        self.ms: float = 0.0

    def __enter__(self) -> "Timer":
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc) -> None:
        self.ms = (time.perf_counter() - self._t0) * 1000.0


# --- Kaggle additions --------------------------------------------------------------
# The repo scores the four groups inline in NB2 and again in NB5. Two copies of the
# same arithmetic is how (a) and (c) end up scored slightly differently — and the whole
# lab is a comparison. One function, called once per version.

def field_accuracy(preds: list[str], records: list[dict],
                   keys: list[str] | None = None) -> dict:
    """Per-field accuracy, plus how often the output parsed at all.

    This is the autopsy view. A single `target` number cannot tell you whether a run
    lost 0.25 because it emits invalid JSON or because it guesses `urgency` from tone
    words, and those two failures have nothing to do with each other. `unparsed` is
    reported separately because a run that emits prose scores 0 on every field for one
    reason, not four.
    """
    keys = keys or TRIAGE_KEYS
    hits = {k: 0 for k in keys}
    counted = {k: 0 for k in keys}
    unparsed = 0
    for pred, rec in zip(preds, records):
        label = rec.get("label")
        if label is None:
            label = _parse_json_loose(rec.get("output", "")) or {}
        obj = _parse_json_loose(pred)
        if not isinstance(obj, dict):
            unparsed += 1
            for k in keys:
                counted[k] += 1
            continue
        for k in keys:
            want = label.get(k)
            if want is None:
                continue
            counted[k] += 1
            got = obj.get(k)
            if got is None:
                continue
            if k == "product":
                hits[k] += int(normalize(str(got)) == normalize(str(want)))
            else:
                hits[k] += int(str(got).strip().lower() == str(want).strip().lower())
    out = {k: (hits[k] / counted[k] if counted[k] else None) for k in keys}
    out["unparsed"] = unparsed
    out["n"] = len(preds)
    return out


def score_version(
    target_records: list[dict],
    target_preds: list[str],
    regression_records: list[dict],
    regression_preds: list[str],
    latency_ms: float,
    extra: dict | None = None,
) -> GroupScores:
    """All four groups for one version, from already-generated strings.

    * target     — mean `triage_field_accuracy` against the labelled JSON
    * regression — mean `keyword_recall` over the general-question keywords
    * format     — mean `has_required_keys`, i.e. "is it the 4-key object we asked for"
    * latency    — passed in, because only the generation loop can measure it honestly

    `format` deliberately uses `has_required_keys` rather than `json_parses`: valid JSON
    with the wrong keys is not format compliance, it just parses. `json_parses` is
    recorded in `extra` so the difference between "not JSON" and "wrong JSON" stays
    visible.
    """
    if len(target_preds) != len(target_records):
        raise ValueError(
            f"target: {len(target_preds)} predictions for {len(target_records)} records. "
            "Scoring misaligned predictions produces a number that means nothing."
        )
    if len(regression_preds) != len(regression_records):
        raise ValueError(
            f"regression: {len(regression_preds)} predictions for "
            f"{len(regression_records)} records."
        )

    tgt, fmt, parses = [], [], []
    for pred, rec in zip(target_preds, target_records):
        label = rec.get("label")
        if label is None:
            label = _parse_json_loose(rec.get("output", "")) or {}
        tgt.append(triage_field_accuracy(pred, label))
        fmt.append(has_required_keys(pred, TRIAGE_KEYS))
        parses.append(json_parses(pred))

    reg = [keyword_recall(p, r.get("keywords", []))
           for p, r in zip(regression_preds, regression_records)]

    ex = {
        "json_parses": round(sum(parses) / len(parses), 4) if parses else 0.0,
        "fields": field_accuracy(target_preds, target_records),
        "n_regression": len(reg),
    }
    if extra:
        ex.update(extra)
    return GroupScores(
        target=sum(tgt) / len(tgt) if tgt else 0.0,
        regression=sum(reg) / len(reg) if reg else 0.0,
        format=sum(fmt) / len(fmt) if fmt else 0.0,
        latency_ms=latency_ms,
        n=len(target_records),
        extra=ex,
    )

In [ ]:
%%writefile labkit/modeling.py
"""Target-module resolution and LoRA parameter accounting.

Two things here are lab-specific and worth reading before you use them.

**1. "all-linear" is a lie on a multimodal base.**
Qwen3.5 ships as `Qwen3_5ForConditionalGeneration`: a text decoder *plus* a vision
tower. PEFT's `target_modules="all-linear"` walks every `nn.Linear` in the model, so
it happily attaches adapters to the vision encoder you are not training on. You get a
bigger adapter, slower steps, and a checkpoint that is wrong to merge. The deck says
"all-linear" (§10.2) because the study behind it used text-only models — on a 2026
multimodal checkpoint you want *all text-decoder linear layers*, which is what
`resolve_target_modules(model, "text-linear")` returns.

**2. Fair contrasts need matched parameter counts, not matched ranks.**
The claim in §10.2 is that attention-only placement loses to full placement *at the
same parameter budget*. Comparing `q,v @ r=16` against `all-linear @ r=16` compares
budgets, not placements, and proves nothing. `matched_rank()` solves for the rank that
puts attention-only on the same budget, so the only variable left is *where*.
"""
from __future__ import annotations

import re
from dataclasses import dataclass

# Vision / multimodal submodules to keep adapters away from when fine-tuning text.
_VISION_HINTS = ("visual", "vision_tower", "vision_model", "image_", "patch_embed", "merger")

# Attention projections, by the names the Qwen3.5 text decoder uses.
_ATTN_SUFFIXES = ("q_proj", "k_proj", "v_proj", "o_proj")
_QV_SUFFIXES = ("q_proj", "v_proj")


@dataclass(frozen=True)
class LinearInfo:
    name: str
    in_features: int
    out_features: int

    @property
    def lora_params_at(self) -> int:  # per unit of rank
        return self.in_features + self.out_features


def iter_linear_modules(model) -> list[LinearInfo]:
    """Every nn.Linear in the model, with its shape. Import-light: no torch type import."""
    out: list[LinearInfo] = []
    for name, mod in model.named_modules():
        if mod.__class__.__name__ not in ("Linear", "Linear4bit", "Params4bit", "LoraLinear"):
            # bitsandbytes/peft wrap Linear; match on the attribute shape instead.
            if not (hasattr(mod, "in_features") and hasattr(mod, "out_features")):
                continue
        if not (hasattr(mod, "in_features") and hasattr(mod, "out_features")):
            continue
        out.append(LinearInfo(name, int(mod.in_features), int(mod.out_features)))
    return out


def is_vision(name: str) -> bool:
    lowered = name.lower()
    return any(hint in lowered for hint in _VISION_HINTS)


def is_head(name: str) -> bool:
    """The LM head / embeddings are excluded — adapting them is a different decision."""
    tail = name.split(".")[-1]
    return tail in ("lm_head", "embed_tokens", "score", "classifier")


def resolve_target_modules(model, mode: str = "text-linear") -> list[str]:
    """Return the *suffix* names PEFT should target, for the given placement mode.

    mode="text-linear" -> every linear layer of the text decoder (the deck's
                          "all-linear", corrected for the vision tower)
    mode="attn-only"   -> q_proj and v_proj only (the 2024-era default; Mistake #1)
    mode="attn-full"   -> q,k,v,o (attention, but complete)
    """
    suffixes: set[str] = set()
    for lin in iter_linear_modules(model):
        if is_vision(lin.name) or is_head(lin.name):
            continue
        tail = lin.name.split(".")[-1]
        if mode == "attn-only":
            if tail in _QV_SUFFIXES:
                suffixes.add(tail)
        elif mode == "attn-full":
            if tail in _ATTN_SUFFIXES:
                suffixes.add(tail)
        elif mode == "text-linear":
            suffixes.add(tail)
        else:
            raise ValueError(f"unknown placement mode {mode!r}")
    if not suffixes:
        raise RuntimeError(
            f"resolve_target_modules found nothing for mode={mode!r}. "
            "Print `[n for n,_ in model.named_modules()]` and check the layer names."
        )

    # PEFT matches target_modules by *suffix*. If a vision-tower layer happens to end
    # in one of the same names (e.g. a vision block that also calls its projection
    # `q_proj`), returning suffixes would silently adapt the vision tower anyway.
    # Detect that collision and fall back to fully-qualified names.
    collides = any(
        is_vision(lin.name) and lin.name.split(".")[-1] in suffixes
        for lin in iter_linear_modules(model)
    )
    if collides:
        return sorted(
            lin.name
            for lin in iter_linear_modules(model)
            if not is_vision(lin.name)
            and not is_head(lin.name)
            and lin.name.split(".")[-1] in suffixes
        )
    return sorted(suffixes)


def count_lora_params(model, target_suffixes: list[str], r: int) -> int:
    """Analytic trainable-parameter count for a LoRA at rank r on those suffixes.

    LoRA adds B (out x r) and A (r x in) per targeted layer => r * (in + out).
    """
    targets = set(target_suffixes)
    total = 0
    for lin in iter_linear_modules(model):
        if is_vision(lin.name) or is_head(lin.name):
            continue
        if lin.name.split(".")[-1] in targets:
            total += r * lin.lora_params_at
    return total


def matched_rank(model, base_suffixes: list[str], base_r: int, other_suffixes: list[str]) -> int:
    """Rank that puts `other_suffixes` on the same parameter budget as base at base_r.

    Returns at least 1. Rounds to the nearest integer rank — exact matching is
    impossible when the layer shapes differ, so report the achieved counts too.
    """
    budget = count_lora_params(model, base_suffixes, base_r)
    per_rank = count_lora_params(model, other_suffixes, 1)
    if per_rank == 0:
        raise RuntimeError(f"no layers matched {other_suffixes!r}")
    return max(1, round(budget / per_rank))


def describe_placement(model, r: int = 16) -> list[dict]:
    """Table for NB4: placement x rank x parameter count. Print this before training."""
    text = resolve_target_modules(model, "text-linear")
    qv = resolve_target_modules(model, "attn-only")
    qv_r = matched_rank(model, text, r, qv)
    rows = [
        {"placement": "text-linear", "modules": len(text), "r": r,
         "trainable": count_lora_params(model, text, r)},
        {"placement": "attn-only(q,v)", "modules": len(qv), "r": r,
         "trainable": count_lora_params(model, qv, r)},
        {"placement": "attn-only(q,v) matched", "modules": len(qv), "r": qv_r,
         "trainable": count_lora_params(model, qv, qv_r)},
    ]
    return rows


def layer_type_summary(model_config) -> dict:
    """Deck §6.4 made visible: the 2026 bases interleave linear and full attention.

    Qwen3.5's text config carries `layer_types` / `full_attention_interval`. Printing
    this in the lab is the point where the architecture section stops being a slide.
    """
    cfg = getattr(model_config, "text_config", model_config)
    types = getattr(cfg, "layer_types", None)
    summary: dict = {
        "num_hidden_layers": getattr(cfg, "num_hidden_layers", None),
        "full_attention_interval": getattr(cfg, "full_attention_interval", None),
        "linear_num_key_heads": getattr(cfg, "linear_num_key_heads", None),
    }
    if types:
        counts: dict[str, int] = {}
        for t in types:
            counts[str(t)] = counts.get(str(t), 0) + 1
        summary["layer_types"] = counts
    return summary

In [ ]:
%%writefile labkit/train.py
"""Thin, version-defensive wrapper around TRL's SFTTrainer.

Why a wrapper at all: the lab this replaces shipped a page of monkey-patches for
`tokenizer=` vs `processing_class=`, `evaluation_strategy` vs `eval_strategy`, and a
`packing=False` workaround. Those were all *pre-1.0 TRL fossils*. Rather than ship a
new set of fossils, this module asks the installed TRL what it accepts and drops what
it does not — so the lab keeps running when TRL moves again, and tells you what it
dropped instead of failing at step 0.

The defaults encode the deck:
  * `target_modules` = text-decoder linear layers (§10.2, corrected for the vision tower)
  * `learning_rate`  = ~10x the full-FT scale (§10.3)
  * effective batch  < 32 (§10.4)
  * the loss mask comes from `labkit.data.to_training_dataset()`, NOT from TRL's
    `assistant_only_loss` — that flag silently supervises nothing on templates without
    `{% generation %}` markers, which includes Qwen3.5 (see check_mask_agreement.py)
  * `loss_type="chunked_nll"` (TRL >= 1.7 default; ~30-50% less VRAM)
"""
from __future__ import annotations

import dataclasses
import inspect
import math
import warnings

from . import device
from .config import MAX_EFFECTIVE_BATCH, LoraSpec, Tier

WARMUP_FRACTION = 0.1


def planned_steps(n_examples: int, tier: Tier, epochs: float) -> int:
    """Optimizer steps that `epochs` over `n_examples` takes on `tier`.

    NB3 sets an *epoch* budget and lets the Trainer derive the step count; NB4's
    contrasts set `max_steps` directly. The autopsy only means anything if both land on
    the SAME number, so both go through this function rather than one of them hardcoding
    a guess. `n_examples` must be the count AFTER `data.to_training_dataset()`, which
    drops examples with zero supervised tokens.
    """
    per_epoch = math.ceil(n_examples / tier.effective_batch)
    return max(1, math.ceil(per_epoch * epochs))


def _accepted_fields(cls) -> set[str]:
    """Field names `cls` will accept, whether it is a dataclass or a plain __init__."""
    names: set[str] = set()
    if dataclasses.is_dataclass(cls):
        names |= {f.name for f in dataclasses.fields(cls)}
    try:
        sig = inspect.signature(cls.__init__)
        names |= {p for p in sig.parameters if p != "self"}
    except (TypeError, ValueError):  # pragma: no cover - builtins
        pass
    return names


def filter_kwargs(cls, desired: dict, *, label: str = "config") -> tuple[dict, list[str]]:
    """Keep only the kwargs `cls` accepts. Returns (kept, dropped_names).

    Dropped keys are reported, never swallowed: if your TRL is too old for
    `padding_free`, you want to *know* that packing is now unsafe, not discover it in
    a loss curve.
    """
    ok = _accepted_fields(cls)
    if not ok:                                    # pragma: no cover - defensive
        return dict(desired), []
    kept = {k: v for k, v in desired.items() if k in ok}
    dropped = sorted(set(desired) - set(kept))
    if dropped:
        warnings.warn(
            f"{label}: installed {cls.__name__} does not accept {dropped}. "
            "They were dropped. Check your TRL/PEFT version against requirements.txt.",
            RuntimeWarning,
            stacklevel=2,
        )
    return kept, dropped


def sft_config_kwargs(
    tier: Tier,
    spec: LoraSpec,
    output_dir: str,
    *,
    max_steps: int | None = None,
    total_steps: int | None = None,
    num_train_epochs: float = 1.0,
    mask_mode: str = "assistant-only",
    seed: int = 42,
    precision: str | None = None,
    eval_steps: int | None = None,
) -> dict:
    """The SFTConfig we *want*. Pass through `filter_kwargs` before constructing.

    Kaggle delta — `eval_steps`. When set (and a `eval_dataset=` is handed to the
    Trainer), held-out loss is logged during training instead of being inferred after
    the fact from the training curve. The repo runs 30 optimizer steps over ~100
    examples at LR 1e-4; whether that is under- or over-fit is currently an argument,
    and the val split it needs has existed since NB1 and gone unused. `eval_strategy`
    is spelled the transformers-v5 way and goes through `filter_kwargs`, so an older
    install drops it with a warning rather than dying at trainer construction.
    """
    if tier.effective_batch > MAX_EFFECTIVE_BATCH:
        raise ValueError(
            f"effective batch {tier.effective_batch} exceeds {MAX_EFFECTIVE_BATCH} "
            "(deck §10.4: LoRA tolerates large batches worse than full FT, and raising "
            "rank does not fix it). Lower grad_accum for this tier."
        )
    kw = dict(
        output_dir=output_dir,
        max_length=tier.max_length,               # NOT max_seq_length (renamed in TRL v1)
        per_device_train_batch_size=tier.per_device_batch,
        gradient_accumulation_steps=tier.grad_accum,
        learning_rate=spec.lr,
        lr_scheduler_type="cosine",
        num_train_epochs=num_train_epochs,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        seed=seed,
        packing=False,       # we supply pre-tokenized labels -- see the note below
        loss_type="chunked_nll",                  # TRL >= 1.7 default; big VRAM saving
        gradient_checkpointing=True,
    )
    # `warmup_ratio` does not exist any more. transformers v5 / TRL 1.10 expose only
    # `warmup_steps` (measured on Colab 2026-08-20: SFTConfig warm-fields == ['warmup_steps']).
    # Passing the ratio does not raise -- filter_kwargs drops it with a warning and the run
    # silently trains with NO warmup, which is exactly the class of quiet failure this lab
    # is about. Convert the deck's 10% into an absolute step count.
    steps = max_steps if max_steps is not None else total_steps
    if steps:
        kw["warmup_steps"] = max(1, round(WARMUP_FRACTION * steps))

    # Precision follows the DEVICE, not the fashion. A free-Colab T4 is Turing and has
    # no bf16 at all — see labkit/device.py. Setting the wrong one here either errors at
    # trainer construction or silently trains in fp32.
    prec = device.precision(precision)
    kw["bf16"] = prec == "bf16"
    kw["fp16"] = prec == "fp16"

    # `padding_free` is enabled only where it is both safe and useful — see
    # device.supports_padding_free(). On the default T4 it is neither: Turing has no
    # FlashAttention-2, and per_device_batch=1 leaves no padding to remove.
    # It also conflicts with our setup: TRL raises
    #   "When padding_free=True without packing, max_length is not enforced"
    # because we pass pre-tokenized labels (packing off). Our inputs ARE already
    # truncated by build_example(), so when padding-free IS available we hand TRL
    # max_length=None to satisfy that check honestly rather than silencing it.
    kw["padding_free"] = device.supports_padding_free(tier.per_device_batch)
    if kw["padding_free"]:
        kw["max_length"] = None

    # NOTE — deliberately NOT setting `assistant_only_loss`.
    # TRL derives that mask from `{% generation %}` markers in the chat template, and
    # Qwen3.5's template has none: the flag produces a mask of ZERO supervised tokens
    # while emitting only a warning. See scripts/check_mask_agreement.py.
    # Instead the dataset is pre-tokenized by labkit.data.to_training_dataset(), so the
    # loss covers exactly the mask verified in NB1. That also forces packing off:
    # packing concatenates examples and would invalidate the label alignment.
    if max_steps is not None:
        kw["max_steps"] = max_steps

    # Held-out loss during training (Kaggle delta). `eval_strategy` is the v5 name;
    # `evaluation_strategy` was the v4 one and filter_kwargs will drop whichever the
    # installed version does not know.
    if eval_steps:
        kw["eval_strategy"] = "steps"
        kw["eval_steps"] = int(eval_steps)
        kw["per_device_eval_batch_size"] = tier.per_device_batch
    return kw


def align_trainable_precision(model, precision: str | None = None) -> dict:
    """Make the trainable params something fp16's GradScaler can actually unscale.

    Measured on a free-Colab T4 (`scripts/probe_precision.py --trainer qlora`): the
    model handed to SFTTrainer has NO bfloat16 parameters, and the model SFTTrainer
    hands back has 496 of them -- every LoRA weight -- while `fp16=True` and a
    GradScaler is attached. Training then dies at the first optimizer step with

        NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda"
                             not implemented for 'BFloat16'

    because that CUDA kernel has no BFloat16 overload. On a Turing card there is no
    bf16 hardware at all, so the cast is not just unsupported, it is meaningless.

    This is the exact failure `labkit/device.py` was written about -- "tutorials hardcode
    bf16 because every 2026 tutorial is written on an A100" -- except the hardcoding is
    inside the training library, downstream of both the quantization config and the
    `fp16`/`bf16` flags we set. So it cannot be fixed by configuring TRL; it has to be
    corrected on the model TRL returns.

    fp32 is the target, not fp16: master weights in fp32 with fp16 autocast is the
    standard mixed-precision setup, and it is what `prepare_model_for_kbit_training`
    produces on its own. Call this after constructing the Trainer and before `.train()`
    -- the optimizer is not built until then, so re-typing the parameters is safe.

    Returns a summary of what moved, so a run that needed the fix says so out loud
    instead of quietly working for reasons nobody can see.
    """
    import torch

    prec = device.precision(precision)
    if prec != "fp16":
        return {"precision": prec, "recast": 0}

    moved = 0
    for param in model.parameters():
        if param.requires_grad and param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float32)
            moved += 1
    total = sum(1 for p in model.parameters() if p.requires_grad)
    return {"precision": prec, "recast": moved, "trainable_tensors": total}


def lora_config_kwargs(spec: LoraSpec, target_modules: list[str]) -> dict:
    if spec.r is None or spec.alpha is None:
        raise ValueError(
            f"spec {spec.key!r} has an unresolved rank. Call "
            "`spec.resolved(modeling.matched_rank(...))` first — see NB4."
        )
    return dict(
        r=spec.r,
        lora_alpha=spec.alpha,                    # §9.3 invariant: alpha = 2r
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=target_modules,
    )


def losses_from_log(log_history: list[dict]) -> dict:
    """Pull the loss curves out of `trainer.state.log_history`.

    TRL logs training loss under `loss` and eval loss under `eval_loss`, in the same
    flat list, one dict per logging event. Returns both curves plus the last value of
    each, so a run row can record them and the report can say "train fell, held-out
    rose" — the shape of overfitting — instead of guessing from the training curve
    alone.
    """
    train_curve = [(e.get("step"), e["loss"]) for e in log_history if "loss" in e]
    eval_curve = [(e.get("step"), e["eval_loss"]) for e in log_history if "eval_loss" in e]
    return {
        "train_curve": train_curve,
        "eval_curve": eval_curve,
        "final_train_loss": train_curve[-1][1] if train_curve else None,
        "final_eval_loss": eval_curve[-1][1] if eval_curve else None,
        "min_eval_loss": min((v for _, v in eval_curve), default=None),
        "eval_rose_after_min": (
            bool(eval_curve) and eval_curve[-1][1] > min(v for _, v in eval_curve) + 1e-9
        ),
    }


def summarize_run(spec: LoraSpec, tier: Tier, target_modules: list[str],
                  trainable: int, seconds: float, peak_vram_gb: float | None,
                  *, max_steps: int | None = None, n_train_examples: int | None = None,
                  final_train_loss: float | None = None,
                  final_eval_loss: float | None = None,
                  replay_fraction: float | None = None) -> dict:
    """One row of `results/runs.csv`. Same shape for every run so runs are comparable.

    Kaggle delta — the last five fields. `max_steps` and `n_train_examples` are what
    make "every run got the same step budget" a checkable fact in the CSV rather than a
    claim in the report; `replay_fraction` is what distinguishes `correct` from
    `correct_replay` at a glance, since every LoRA column is identical between them;
    and the two losses let the report separate "trained longer" from "fit better".
    Rows with new keys are unioned into the header by `report.append_row`, so a mixed
    CSV from an older run still loads.
    """
    return {
        "run": spec.key,
        "label": spec.label,
        "tier": tier.name,
        "model": tier.model_id,
        # Recorded because wall-clock numbers are NOT comparable across precisions, and
        # this lab has already shipped one set of timings measured on a path (emulated
        # bf16 on a T4) that was later removed. A row without its precision is a row you
        # cannot compare to anything.
        "precision": device.precision(),
        "placement": spec.target,
        "n_target_modules": len(target_modules),
        "r": spec.r,
        "lora_alpha": spec.alpha,
        "learning_rate": spec.lr,
        "load_in_4bit": spec.load_in_4bit,
        "trainable_params": trainable,
        "max_steps": max_steps,
        "n_train_examples": n_train_examples,
        "replay_fraction": replay_fraction,
        "final_train_loss": None if final_train_loss is None else round(final_train_loss, 4),
        "final_eval_loss": None if final_eval_loss is None else round(final_eval_loss, 4),
        "train_seconds": round(seconds, 1),
        "peak_vram_gb": None if peak_vram_gb is None else round(peak_vram_gb, 2),
    }

In [ ]:
%%writefile labkit/generate.py
"""Model loading and batch generation — the only GPU-touching module.

Kept separate from `evaluate` on purpose: scoring is pure functions over strings and
runs anywhere, so the whole grading contract stays testable on a laptop. Only this
file needs a GPU.
"""
from __future__ import annotations

import gc
import time

from . import device
from .config import NAIVE_PROMPT, OPTIMIZED_PROMPT, Tier

# The two prompts that define baselines (a) and (b). Baseline (b) has to be a genuine
# effort — deck §17's whole point is that a fine-tune which cannot beat a *well-prompted*
# base model is not worth shipping. Writing a deliberately weak (b) to flatter your
# fine-tune is the main way to cheat this lab, and the rubric checks for it.

def free_memory() -> None:
    """Between runs. Deck §16: not doing this is the most common OOM in a multi-run lab."""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
    except ImportError:      # pragma: no cover
        pass


def peak_vram_gb() -> float | None:
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.max_memory_allocated() / 1024 ** 3
    except ImportError:      # pragma: no cover
        pass
    return None


def load_base(tier: Tier, load_in_4bit: bool = False):
    """Load the base model + tokenizer for `tier`.

    `load_in_4bit` is exposed only so NB4 can *measure* the QLoRA contrast. The default
    is bf16 because the vendor advises against 4-bit on this model family (deck §12).
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(tier.model_id, trust_remote_code=True)
    # dtype (not torch_dtype — deprecated in transformers 5.x) and NOT hardcoded bf16:
    # the lab's default tier is a T4, which has no bfloat16 (see labkit/device.py).
    kwargs: dict = {"trust_remote_code": True, "dtype": device.torch_dtype(),
                    "device_map": "auto"}
    if load_in_4bit:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=device.torch_dtype(),
        )
    model = AutoModelForCausalLM.from_pretrained(tier.model_id, **kwargs)
    return model, tok


def render_prompt(tok, prompt: str, system: str | None,
                  enable_thinking: bool | None = False) -> str:
    """The exact string handed to the model. Shared by generation and by cost accounting
    so "prompt tokens" is measured on what was actually sent, not on a re-render."""
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    kw: dict = {"tokenize": False, "add_generation_prompt": True}
    # None means OMIT the kwarg, matching data._render(). Passing a literal None
    # is not the same thing: it lands in the Jinja context where `is defined` is
    # true, so the template may read it as falsey and quietly behave like False.
    # Training and generation have to be able to express "template default"
    # identically or they cannot be compared at all.
    if enable_thinking is not None:
        kw["enable_thinking"] = enable_thinking
    try:
        return tok.apply_chat_template(msgs, **kw)
    except TypeError:
        kw.pop("enable_thinking", None)
        return tok.apply_chat_template(msgs, **kw)


def generate_measured(
    model,
    tok,
    prompts: list[str],
    system: str | None = None,
    max_new_tokens: int = 160,
    enable_thinking: bool | None = False,
    batch_size: int = 4,
    label: str = "generate",
    progress: bool = True,
    warmup: bool = True,
) -> tuple[list[str], dict]:
    """Greedy decode. Returns (completions, stats).

    Greedy (do_sample=False) is deliberate: this lab compares runs, and sampling noise
    would swamp the differences you are trying to measure.

    Kaggle edition, two additions over `src/labkit`:

    * **`warmup=True` discards one throwaway generation before the clock starts.** The
      first `generate()` on a freshly loaded model pays for CUDA kernel autotuning,
      cuBLAS workspace allocation and the first KV-cache malloc. Measured on a T4 that
      first call is seconds slower than the second, and with the shipped eval sizes it
      lands almost entirely in batch 1 — i.e. straight into the latency number the cost
      section then multiplies by 1000. Warm-up cost is real but it is a *deployment*
      cost paid once, not a per-ticket cost.
    * **token accounting.** Latency alone cannot distinguish "the fine-tune is faster"
      from "the fine-tune emitted fewer tokens", and for this lab it is nearly always
      the latter: the base model with prompt (b) writes an explanation, the fine-tune
      writes 40 tokens of JSON. `ms_per_new_token` separates the two, and
      `prompt_tokens_mean` is what makes the prompt-shrink argument in `cost.py` an
      economic claim instead of an aesthetic one.

    `progress` prints a per-batch line with an ETA. This is not decoration: on a free
    T4 a 4B model takes tens of minutes to score the eval set, and a notebook that
    prints nothing for that long is indistinguishable from a hang. Students kill runs
    that look stuck.
    """
    import torch

    outs: list[str] = []
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    texts_all = [render_prompt(tok, p, system, enable_thinking) for p in prompts]
    prompt_tok_counts = [len(tok(t, add_special_tokens=False)["input_ids"])
                         for t in texts_all]

    warmup_ms = 0.0
    if warmup and prompts:
        enc = tok(texts_all[:1], return_tensors="pt", padding=True).to(model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            model.generate(**enc, max_new_tokens=8, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        warmup_ms = (time.perf_counter() - t0) * 1000.0
        if progress:
            print(f"  [{label}] warm-up discarded: {warmup_ms:.0f} ms", flush=True)

    total_ms = 0.0
    new_tokens = 0
    n_batches = (len(prompts) + batch_size - 1) // batch_size
    t_start = time.perf_counter()
    for bi, i in enumerate(range(0, len(prompts), batch_size), start=1):
        texts = texts_all[i:i + batch_size]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tok.pad_token_id,
            )
        total_ms += (time.perf_counter() - t0) * 1000.0
        for row, src in zip(gen, enc["input_ids"]):
            tail = row[len(src):]
            # Count real tokens only: padded rows finish early and the rest of the row
            # is pad. Counting pad as work would make a batch look slower per token the
            # more uneven it is.
            kept = [int(t) for t in tail.tolist() if t != tok.pad_token_id]
            new_tokens += len(kept)
            outs.append(tok.decode(tail, skip_special_tokens=True).strip())

        if progress:
            done = time.perf_counter() - t_start
            eta = done / bi * (n_batches - bi)
            print(f"  [{label}] batch {bi}/{n_batches}  "
                  f"{done:5.0f}s elapsed  ~{eta:5.0f}s left", flush=True)

    n = max(1, len(prompts))
    stats = {
        "label": label,
        "n_prompts": len(prompts),
        "batch_size": batch_size,
        "max_new_tokens": max_new_tokens,
        "warmup_ms": round(warmup_ms, 1),
        "gen_seconds": round(total_ms / 1000.0, 2),
        "mean_latency_ms": round(total_ms / n, 1),
        "prompt_tokens_total": sum(prompt_tok_counts),
        "prompt_tokens_mean": round(sum(prompt_tok_counts) / n, 1),
        "new_tokens_total": new_tokens,
        "new_tokens_mean": round(new_tokens / n, 1),
        "ms_per_new_token": round(total_ms / new_tokens, 2) if new_tokens else None,
        "tokens_per_second": round(new_tokens / (total_ms / 1000.0), 1) if total_ms else None,
    }
    if progress:
        print(f"  [{label}] done: {len(prompts)} prompts in "
              f"{time.perf_counter() - t_start:.0f}s  "
              f"({stats['mean_latency_ms']:.0f} ms/sample, "
              f"{stats['new_tokens_mean']:.0f} new tok/sample)", flush=True)
    return outs, stats


def generate_batch(
    model,
    tok,
    prompts: list[str],
    system: str | None = None,
    max_new_tokens: int = 160,
    enable_thinking: bool | None = False,
    batch_size: int = 4,
    label: str = "generate",
    progress: bool = True,
    warmup: bool = False,
) -> tuple[list[str], float]:
    """`generate_measured` with the repo's original 2-tuple signature, so the six
    stage notebooks keep working unchanged. New code should call `generate_measured`
    and keep the whole stats dict — the cost section needs the token counts."""
    outs, stats = generate_measured(
        model, tok, prompts, system=system, max_new_tokens=max_new_tokens,
        enable_thinking=enable_thinking, batch_size=batch_size, label=label,
        progress=progress, warmup=warmup,
    )
    return outs, stats["mean_latency_ms"]

In [ ]:
%%writefile labkit/cost.py
"""Latency + prompt length -> money. The missing half of "was the fine-tune worth it?"

The lab measures three versions and picks a winner on accuracy. That is only half the
decision a team actually makes. The other half is the one the deck's §17 framing asks
for: a fine-tune costs GPU-hours up front and buys you a *shorter prompt* and *shorter
output* on every request forever. This module turns the numbers the eval sweep already
produced into the two figures a decision needs:

    $/1k tickets      what serving this version costs
    break-even        how many tickets it takes to repay the training run

**Everything here is arithmetic over measured inputs.** `mean_latency_ms`,
`prompt_tokens_mean` and `new_tokens_mean` come from `generate.generate_measured()`;
the prices are stated assumptions with a source. Nothing is estimated from a vibe, and
every price is a keyword argument so a report can re-run the whole section against its
own cloud bill instead of arguing with the defaults.

**The honest caveats, stated once and re-stated in the notebook output:**

1. `mean_latency_ms` is *batched* throughput at the eval sweep's batch size. Serving a
   single ticket interactively is slower per ticket; serving at batch 32 is faster.
   The comparison BETWEEN versions is fair because every version is measured at the
   same batch size, which is the claim this lab needs. It is not a quote for prod.
2. Self-hosted cost assumes the GPU is busy. A GPU rented and left idle costs the same
   per hour, so the self-hosted numbers are a floor that only a well-fed queue reaches.
3. Nothing here prices engineer time, and the fine-tune spent some.
"""
from __future__ import annotations

# --- price assumptions -------------------------------------------------------------
# Rented-GPU rate. The default is deliberately the *cheap* end of the market for a T4
# class card (spot/preemptible, mid-2026, ~$0.35/hr) because a low self-host rate is the
# assumption HOSTILE to the conclusion this lab wants to draw: it makes fine-tuning look
# expensive to amortize, not cheap. If your bill says otherwise, pass your own number.
GPU_HOURLY_USD = 0.35

# A hosted small-model API, priced per 1M tokens, as the "don't fine-tune anything"
# alternative. Order of magnitude of a 2026 small/flash tier.
API_INPUT_USD_PER_MTOK = 0.15
API_OUTPUT_USD_PER_MTOK = 0.60


def self_hosted_per_1k(mean_latency_ms: float,
                       gpu_hourly_usd: float = GPU_HOURLY_USD) -> float:
    """$ to serve 1000 tickets on a GPU you rent by the hour."""
    gpu_hours = (mean_latency_ms / 1000.0) * 1000 / 3600.0
    return gpu_hours * gpu_hourly_usd


def api_per_1k(prompt_tokens_mean: float, new_tokens_mean: float,
               input_usd_per_mtok: float = API_INPUT_USD_PER_MTOK,
               output_usd_per_mtok: float = API_OUTPUT_USD_PER_MTOK) -> float:
    """$ to serve 1000 tickets through a token-priced API at these token counts.

    This is why prompt length is an economic quantity: prompt (b) carries the whole
    schema on *every* request, and a fine-tune moves that schema into the weights.
    """
    return (prompt_tokens_mean * 1000 / 1e6) * input_usd_per_mtok + \
           (new_tokens_mean * 1000 / 1e6) * output_usd_per_mtok


def training_usd(train_seconds: float, gpu_hourly_usd: float = GPU_HOURLY_USD) -> float:
    """One-off cost of the training run itself."""
    return (train_seconds / 3600.0) * gpu_hourly_usd


def break_even_tickets(training_cost_usd: float, saving_per_1k_usd: float) -> float | None:
    """How many tickets before the training run has paid for itself.

    `None` when the saving is zero or negative — the honest answer to "when does this
    pay off?" is sometimes "never", and a number like 1e9 hides that.
    """
    if saving_per_1k_usd <= 0:
        return None
    return training_cost_usd / saving_per_1k_usd * 1000


def row(version: str, stats: dict, *, gpu_hourly_usd: float = GPU_HOURLY_USD,
        input_usd_per_mtok: float = API_INPUT_USD_PER_MTOK,
        output_usd_per_mtok: float = API_OUTPUT_USD_PER_MTOK) -> dict:
    """One line of the cost table, straight from a `generate_measured` stats dict."""
    lat = float(stats["mean_latency_ms"])
    p_tok = float(stats["prompt_tokens_mean"])
    o_tok = float(stats["new_tokens_mean"])
    return {
        "version": version,
        "prompt_tok": round(p_tok, 1),
        "out_tok": round(o_tok, 1),
        "ms/sample": round(lat, 1),
        "ms/token": stats.get("ms_per_new_token"),
        "self_host_$/1k": round(self_hosted_per_1k(lat, gpu_hourly_usd), 4),
        "api_$/1k": round(api_per_1k(p_tok, o_tok, input_usd_per_mtok,
                                     output_usd_per_mtok), 4),
    }


def compare(stats_by_version: dict[str, dict], *,
            train_seconds: float | None = None,
            baseline: str = "(b) base + optimized prompt",
            winner: str = "(c) fine-tune",
            gpu_hourly_usd: float = GPU_HOURLY_USD,
            input_usd_per_mtok: float = API_INPUT_USD_PER_MTOK,
            output_usd_per_mtok: float = API_OUTPUT_USD_PER_MTOK) -> dict:
    """The whole cost section: per-version rows plus the break-even against `baseline`.

    `baseline` is the *prompted base model*, not the naive prompt, because that is the
    real alternative to fine-tuning — deck §17: a fine-tune that cannot beat a
    well-prompted base model is not worth shipping, and that applies to its bill too.
    """
    rows = [row(v, s, gpu_hourly_usd=gpu_hourly_usd,
                input_usd_per_mtok=input_usd_per_mtok,
                output_usd_per_mtok=output_usd_per_mtok)
            for v, s in stats_by_version.items()]
    by_version = {r["version"]: r for r in rows}
    out: dict = {
        "rows": rows,
        "assumptions": {
            "gpu_hourly_usd": gpu_hourly_usd,
            "api_input_usd_per_mtok": input_usd_per_mtok,
            "api_output_usd_per_mtok": output_usd_per_mtok,
            "note": "mean_latency_ms is batched throughput at the eval sweep's batch "
                    "size, measured with warm-up excluded. Comparable across versions; "
                    "not a production quote.",
        },
    }
    if baseline in by_version and winner in by_version:
        b, w = by_version[baseline], by_version[winner]
        deltas = {
            "baseline": baseline,
            "winner": winner,
            "latency_ms_delta": round(w["ms/sample"] - b["ms/sample"], 1),
            "latency_speedup_x": round(b["ms/sample"] / w["ms/sample"], 2)
            if w["ms/sample"] else None,
            "prompt_tokens_saved": round(b["prompt_tok"] - w["prompt_tok"], 1),
            "output_tokens_saved": round(b["out_tok"] - w["out_tok"], 1),
            "self_host_saving_per_1k": round(b["self_host_$/1k"] - w["self_host_$/1k"], 4),
            "api_saving_per_1k": round(b["api_$/1k"] - w["api_$/1k"], 4),
        }
        if train_seconds is not None:
            tc = training_usd(train_seconds, gpu_hourly_usd)
            deltas["training_usd"] = round(tc, 3)
            deltas["train_seconds"] = round(train_seconds, 1)
            be = break_even_tickets(tc, deltas["self_host_saving_per_1k"])
            deltas["break_even_tickets_self_host"] = None if be is None else int(be)
            be_api = break_even_tickets(tc, deltas["api_saving_per_1k"])
            deltas["break_even_tickets_api_equivalent"] = None if be_api is None else int(be_api)
        out["delta"] = deltas
    return out


def verdict_line(comparison: dict) -> str:
    """One sentence a report can quote. Says "never" when the answer is never."""
    d = comparison.get("delta")
    if not d:
        return "cost comparison unavailable: baseline or winner version missing."
    parts = [
        f"{d['winner']} vs {d['baseline']}: "
        f"{d['latency_ms_delta']:+.0f} ms/sample",
    ]
    if d["prompt_tokens_saved"]:
        parts.append(f"{d['prompt_tokens_saved']:+.0f} prompt tokens")
    if d["output_tokens_saved"]:
        parts.append(f"{d['output_tokens_saved']:+.0f} output tokens")
    parts.append(f"self-host {d['self_host_saving_per_1k']:+.4f} $/1k")
    line = ", ".join(parts) + "."
    if "break_even_tickets_self_host" in d:
        be = d["break_even_tickets_self_host"]
        line += (f" Training cost ${d['training_usd']:.3f}; break-even at "
                 f"{be:,} tickets." if be is not None else
                 f" Training cost ${d['training_usd']:.3f} and serving is NOT cheaper — "
                 "it never breaks even on cost alone; the case has to be made on quality.")
    return line

### Kiểm tra harness ngay: import, tier, và các unit-invariant

Ba thứ được assert ở đây vì cả ba đều là *lỗi im lặng* nếu sai: batch hiệu dụng
vượt 32 (deck §10.4), `alpha != 2r` (§9.3), và LR LoRA đặt sai thang (§10.3).

In [ ]:
import importlib, json, sys
sys.path.insert(0, str(WORK))
for m in list(sys.modules):
    if m == "labkit" or m.startswith("labkit."):
        del sys.modules[m]

import labkit
from labkit import config as C, cost, data, device, evaluate as ev
from labkit import generate, modeling, replay, report, train

TIER = C.get_tier()
assert TIER.effective_batch <= C.MAX_EFFECTIVE_BATCH, "batch hiệu dụng > 32 (§10.4)"
assert C.SPECS["correct"].alpha == 2 * C.SPECS["correct"].r, "alpha != 2r (§9.3)"
assert abs(C.LORA_LR / C.FULL_FT_LR - 10.0) < 1e-9, "LoRA LR không ở thang 10x (§10.3)"

print("labkit", labkit.__version__)
print(device.banner())
print(f"\ntier={TIER.name}  model={TIER.model_id}  max_length={TIER.max_length}  "
      f"batch={TIER.per_device_batch}x{TIER.grad_accum}={TIER.effective_batch}")
print("\nNhững gì bản Kaggle này làm khác repo:")
for what, why in labkit.KAGGLE_DELTAS:
    print(f"\n  • {what}\n      {why}")

## 5. Dữ liệu, chat template và **mask loss** (NB1 của lab)

> Deck §13.2: *che loss và chat template quyết định kết quả nhiều hơn mọi biến thể
> LoRA cộng lại.*

Phần này sinh 4 artefact bắt buộc và **fail ngay** nếu mask sai — trước khi tiêu một
phút GPU nào cho việc train:

1. `template_check.json` — template có nuốt khối `<think>` không (§16)
2. `mask_proof.json` — loss **chứa** câu trả lời và **không chứa** câu hỏi
3. `token_stats.json` — p95 → `max_length` (số đo, không phải số đoán)
4. `data/split/{train,val}.jsonl` — split seed 42, và `val` **được dùng thật** ở §7

In [ ]:
import json, pathlib

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

RESULTS = WORK / "results"
train_raw = load_jsonl(DATA / "train_seed.jsonl")
target_all = load_jsonl(DATA / "eval_target.jsonl")
regression_all = load_jsonl(DATA / "eval_regression.jsonl")
holdout_all = load_jsonl(DATA / "holdout_secret.jsonl")

LIMIT = int(os.environ.get("EVAL_LIMIT", "0") or 0)
target = target_all[:LIMIT] if LIMIT else target_all
regression = regression_all[:LIMIT] if LIMIT else regression_all
holdout = holdout_all[:LIMIT] if LIMIT else holdout_all
print(f"train={len(train_raw)}  target={len(target)}/{len(target_all)}  "
      f"regression={len(regression)}/{len(regression_all)}  "
      f"holdout={len(holdout)}/{len(holdout_all)}")
print(json.dumps(train_raw[0], ensure_ascii=False, indent=2)[:500])

### 5.1 Tokenizer + kiểm tra bắt buộc #1: template có giữ khối suy luận?

Chỉ tải **file tokenizer** (vài MB), chưa tải trọng số.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(TIER.model_id, trust_remote_code=True)
check = data.thinking_survives(tok)
print("eos:", tok.eos_token, "| VERDICT:", check["verdict"])
print("--- chuỗi đã render ---")
print(check["rendered"])
report.write_json(check, "template_check.json", results_dir=RESULTS)

### 5.2 Kiểm tra bắt buộc #2: mask — và **đọc** nó

Bốn chế độ; ở đây in ra hai chế độ đối lập nhất để bạn thấy khác biệt bằng mắt.
`everything` là bug kinh điển: câu hỏi nằm trong loss → model học *viết lại câu hỏi*.

In [ ]:
sample = data.to_messages(train_raw[0])
for mode in ("assistant-only", "everything"):
    ex = data.build_example(tok, sample, max_length=TIER.max_length, mask_mode=mode)
    print("=" * 72)
    print(f"mode={mode}  supervised {ex.n_supervised}/{ex.n_total} "
          f"({ex.supervised_fraction:.0%})")
    print("--- LOSS TÍNH TRÊN ĐOẠN NÀY ---")
    print(data.decode_supervised(tok, ex)[:400])

In [ ]:
ex = data.build_example(tok, sample, max_length=TIER.max_length,
                        mask_mode=MASK_MODE)
supervised = data.decode_supervised(tok, ex)
masked = data.decode_masked(tok, ex)
answer = sample[-1]["content"][:40]
question_fragment = train_raw[0]["input"][:40]

proof = {
    "mask_mode": MASK_MODE,
    "n_supervised": ex.n_supervised,
    "n_total": ex.n_total,
    "supervised_fraction": round(ex.supervised_fraction, 4),
    "answer_is_supervised": answer in supervised,
    "question_is_masked": question_fragment not in supervised,
    "supervised_preview": supervised[:300],
    "masked_preview": masked[:300],
}
assert proof["answer_is_supervised"], "câu trả lời KHÔNG nằm trong loss — mask sai"
assert proof["question_is_masked"], "câu hỏi ĐANG nằm trong loss — mask sai"
assert proof["supervised_fraction"] < 0.95, (
    "gần như mọi token đều vào loss — bạn đang train cả prompt (rubric auto-zero)")
print(json.dumps({k: v for k, v in proof.items() if not k.endswith("preview")},
                 ensure_ascii=False, indent=2))
report.write_json(proof, "mask_proof.json", results_dir=RESULTS)

### 5.3 Kiểm tra bắt buộc #3 (bản Kaggle thêm): **prompt lúc train == prompt lúc chấm**

Đây là lỗi F-31 của lab, và nó không tự báo: nếu chuỗi mà evaluation gửi đi không
phải **tiền tố** của chuỗi đã train, adapter vẫn train xong bình thường rồi ra
`target = 0.000` trên mọi cấu hình. Repo mô tả nó trong docstring; ở đây nó là một
`assert` chạy trên 5 mẫu thật.

In [ ]:
align = data.assert_prompt_alignment(tok, train_raw[:5], system=C.NAIVE_PROMPT)
print(json.dumps({k: v for k, v in align.items()
                  if k not in ("eval_render_example", "supervised_tail_example")},
                 ensure_ascii=False, indent=2))
print("\n--- evaluation sẽ gửi đúng chuỗi này ---")
print(align["eval_render_example"])
print("--- và train supervise đúng phần đuôi này ---")
print(align["supervised_tail_example"][:200])
report.write_json(align, "prompt_alignment.json", results_dir=RESULTS)

### 5.4 `max_length` từ p95, và split seed 42

In [ ]:
lengths = [data.build_example(tok, data.to_messages(r), max_length=8192).n_total
           for r in train_raw]
stats = data.token_stats(lengths)
print(json.dumps(stats, ensure_ascii=False, indent=2))
report.write_json(stats, "token_stats.json", results_dir=RESULTS)
if stats["suggested_max_length"] != TIER.max_length:
    print(f"\n⚠ p95 gợi ý max_length={stats['suggested_max_length']} nhưng tier "
          f"đang dùng {TIER.max_length}. Ghi lựa chọn này vào REPORT.md.")

train_rows, val_rows = data.split(train_raw, train_frac=0.9, seed=SEED)
split_dir = DATA / "split"
split_dir.mkdir(exist_ok=True)
for name, rows in (("train", train_rows), ("val", val_rows)):
    with (split_dir / f"{name}.jsonl").open("w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"\ntrain={len(train_rows)}  val={len(val_rows)}  (seed={SEED})")

## 5b. Corpus replay (deck §14.3) — và **chứng minh** nó không trùng tập regression

Đây là phần cải tiến chính so với pipeline gốc. Repo tự đo được: fine-tune đạt
target 0.990 nhưng general capability rơi **0.644 → 0.067**. Nguyên nhân không phải
LoRA mà là *corpus*: mọi input đều là ticket và mọi output đều là JSON, nên
"có input" và "xuất JSON triage" trở thành cùng một thứ trong mắt model.

Deck §14.3 nói cách chữa là trộn lại 1–5% dữ liệu tổng quát. Rủi ro của cách chữa đó
là **nhiễm tập eval**: nếu dữ liệu trộn vào chứa chính câu hỏi dùng để đo regression
thì điểm regression tăng vì đã học thuộc đề, không phải vì giữ được năng lực. Nên
47 mẫu ở đây được kiểm bằng hai điều kiện — không trùng chính xác (sau khi chuẩn hoá
dấu) và Jaccard từ vựng ≤ 0.6 với **cả 15** câu regression.

In [ ]:
pool = replay.load()
decon = data.assert_replay_decontaminated(pool, regression_all)
print(json.dumps(decon, ensure_ascii=False, indent=2))
print(f"\ncặp gần nhất: replay={decon['worst_pair'][0]!r}")
print(f"              eval={decon['worst_pair'][1]!r}")

mixed_rows, replay_manifest = data.mix_replay(
    train_rows, pool, C.replay_fraction(), seed=SEED)
print("\n" + json.dumps({k: v for k, v in replay_manifest.items()
                          if k != "replay_instructions"},
                         ensure_ascii=False, indent=2))
report.write_json({"decontamination": decon, "mix": replay_manifest},
                  "replay_manifest.json", results_dir=RESULTS)

## 6. Ba phiên bản — và **đo baseline TRƯỚC khi train**

> Deck §17: điểm không nằm ở việc perplexity giảm bao nhiêu, mà ở việc bạn có chứng
> minh được bản fine-tune thắng **(b)** — và có phát hiện được nếu nó *không* thắng.

| | Là gì | Vì sao có mặt |
|---|---|---|
| **(a)** | base + prompt ngây thơ (`"Phân loại ticket sau."`) | mốc sàn |
| **(b)** | base + prompt **đã tối ưu** (schema + enum + 1 ví dụ) | **mốc thật sự phải vượt** |
| **(c)** | fine-tune, chấm với prompt *ngây thơ* | hành vi đã nằm trong trọng số |

Thứ tự có lý do: đo (b) **sau** khi biết điểm fine-tune thì bạn sẽ vô thức hạ (b) cho
tới lúc mình thắng. Nên (a) và (b) được đo và **đóng băng** ở đây, kèm SHA của prompt.

Mỗi phiên bản đi qua **cùng một** `eval_pass`: 50 ticket target · 15 câu tổng quát
(regression) · 20 ticket **holdout** chưa ai thấy · 3 ticket mẫu · 3 câu hỏi thường
ngày. Cùng batch size, cùng `max_new_tokens`, greedy — nên số liệu so được với nhau.

In [ ]:
EVAL_BATCH = 4                 # T4 16 GB, 4B model, max_length 1024
MAX_NEW    = 160

V_A  = "(a) base + naive prompt"
V_B  = "(b) base + optimized prompt"
V_C  = "(c) fine-tune"
V_CR = "(c+) fine-tune + replay"

# Câu hỏi "chạy thử" (§11). Cùng bộ này cho MỌI phiên bản, sinh ngay trong lượt
# nạp model của phiên bản đó — thêm một lần load 4B model chỉ để in 6 câu là
# ~2 phút GPU cho mỗi phiên bản, không đổi lấy thông tin nào.
SAMPLE_TICKETS = [
    ("trong miền", "Mình mua tai nghe bluetooth mã đơn DH998877, hộp còn nguyên "
                   "nhưng nghe một bên rất rè. Mình muốn đổi cái khác, gấp nhé!"),
    ("hai ý xung đột", "Đơn ND221100 giao chậm 5 ngày rồi mà bàn ủi hơi nước lại "
                       "bị móp. Shop hoàn tiền cho mình luôn được không?"),
    ("ngoài miền", "Shop có bán cà phê hạt Arabica không, và giao tới Đà Lạt mất "
                   "bao lâu?"),
]
# Ba câu KHÔNG liên quan tới triage: đây là chỗ quên thảm hoạ (§14.3) hiện ra
# bằng mắt thường, không cần đọc bảng điểm.
SAMPLE_GENERAL = [
    "Thủ đô của Nhật Bản là thành phố nào?",
    "Giải thích ngắn gọn vì sao trời có mưa.",
    "Viết một câu cảm ơn khách hàng đã mua hàng.",
]

SCORES, PREDS, STATS, HOLD = {}, {}, {}, {}

def eval_pass(model, tok, version, system, *, full=True):
    """Một phiên bản, bốn nhóm điểm, cùng một đường đi.

    `full=False` chỉ chấm target+format: ba contrast ở §8 không cần nhóm
    regression (phán quyết được chấm trên `correct`), và mỗi lượt sinh thêm là
    ~3 phút GPU cho mỗi adapter.
    """
    tp, st = generate.generate_measured(
        model, tok, [r["input"] for r in target], system=system,
        max_new_tokens=MAX_NEW, batch_size=EVAL_BATCH, label=f"{version}/target")
    rp, hp, sp, gp = [], [], [], []
    if full:
        rp, _ = generate.generate_measured(
            model, tok, [r["instruction"] for r in regression], system=None,
            max_new_tokens=96, batch_size=EVAL_BATCH, warmup=False,
            label=f"{version}/regression")
        hp, _ = generate.generate_measured(
            model, tok, [r["input"] for r in holdout], system=system,
            max_new_tokens=MAX_NEW, batch_size=EVAL_BATCH, warmup=False,
            label=f"{version}/holdout")
        sp, _ = generate.generate_measured(
            model, tok, [t for _, t in SAMPLE_TICKETS], system=system,
            max_new_tokens=MAX_NEW, batch_size=len(SAMPLE_TICKETS),
            warmup=False, progress=False, label=f"{version}/samples")
        gp, _ = generate.generate_measured(
            model, tok, SAMPLE_GENERAL, system=None, max_new_tokens=96,
            batch_size=len(SAMPLE_GENERAL), warmup=False, progress=False,
            label=f"{version}/general")

    trace = sum(ev.valid_reasoning_trace(p) for p in tp) / max(1, len(tp))
    sc = ev.score_version(
        target, tp, regression if full else [], rp, st["mean_latency_ms"],
        extra={"generation": st, "valid_trace_rate": round(trace, 4),
               "system_prompt": (system or "")[:60], "scored_regression": full})
    SCORES[version] = sc
    STATS[version] = st
    PREDS[version] = {"target": tp, "regression": rp, "holdout": hp,
                      "samples": sp, "general": gp}
    if hp:
        HOLD[version] = {
            "target": round(sum(ev.triage_field_accuracy(p, r["label"])
                                for p, r in zip(hp, holdout)) / len(holdout), 4),
            "fields": ev.field_accuracy(hp, holdout), "n": len(holdout)}
    line = (f"{version:<30} target={sc.target:.3f}  format={sc.format:.3f}  "
            f"{sc.latency_ms:7.0f} ms/mẫu  {st['prompt_tokens_mean']:5.0f} tok prompt  "
            f"{st['new_tokens_mean']:5.0f} tok ra")
    if full:
        line += f"  regression={sc.regression:.3f}  holdout={HOLD[version]['target']:.3f}"
    print("\n" + line + "\n")
    return sc

In [ ]:
import time

base, tok = generate.load_base(TIER)
generate.free_memory()
print(json.dumps(modeling.layer_type_summary(base.config), ensure_ascii=False,
                 indent=2))

t0 = time.perf_counter()
eval_pass(base, tok, V_A, C.NAIVE_PROMPT)
eval_pass(base, tok, V_B, C.OPTIMIZED_PROMPT)
print(f"hai baseline: {time.perf_counter() - t0:.0f}s")

### 6.1 Đóng băng — từ đây không sửa tập eval, không sửa prompt (b)

In [ ]:
import hashlib

frozen = {
    "tier": TIER.name,
    "model": TIER.model_id,
    "baseline_a": SCORES[V_A].as_dict(),
    "baseline_b": SCORES[V_B].as_dict(),
    "optimized_prompt_sha": hashlib.sha256(
        C.OPTIMIZED_PROMPT.encode()).hexdigest()[:16],
    "n_target": len(target),
    "n_regression": len(regression),
    "n_holdout": len(holdout),
    "eval_limit": LIMIT or None,
    "smoke_mode": bool(LIMIT) or TIER.name == "SMOKE",
    "holdout": {k: v for k, v in HOLD.items()},
    "stats": {V_A: STATS[V_A], V_B: STATS[V_B]},
}
report.write_json(frozen, "baselines_frozen.json", results_dir=RESULTS)

print(report.markdown_table(ev.comparison_table(
    {V_A: SCORES[V_A], V_B: SCORES[V_B]})))
d = SCORES[V_B].target - SCORES[V_A].target
print(f"\nprompt tối ưu đổi được {d:+.3f} target với "
      f"{STATS[V_B]['prompt_tokens_mean'] - STATS[V_A]['prompt_tokens_mean']:+.0f} "
      f"token prompt mỗi request — đây là mốc (b), và §10 sẽ tính giá của nó.")
if d <= 0:
    print("⚠ (b) KHÔNG hơn (a). Prompt 'tối ưu' của bạn chưa đủ tốt — sửa NGAY "
          "bây giờ, trước khi train. Thắng một mốc yếu là thắng giả.")

del base
generate.free_memory()

## 7. Huấn luyện — cấu hình ĐÚNG (deck §10), **một hàm cho mọi run**

| Nút | Giá trị | Deck |
|---|---|---|
| `target_modules` | **toàn bộ linear của text decoder** (không gồm vision tower) | §10.2 |
| `learning_rate` | **1e-4 ≈ 10× LR full-FT** | §10.3 |
| batch hiệu dụng | **16 < 32** | §10.4 |
| `alpha` | `2r` | §9.3 |
| `packing` | **tắt** — ta nạp nhãn đã token hoá sẵn, packing sẽ phá mask | §13.3 |
| `padding_free` | chỉ khi có FlashAttention **và** batch ≥ 2 → trên T4 là **không** | §13.3 |
| `loss_type` | `chunked_nll` | §15 |

Mọi run trong notebook này — `correct`, `correct_replay`, và ba contrast ở §8 — đi qua
**đúng một** hàm `train_one()` với **đúng một** ngân sách step. Hai bản sao của cùng
một vòng train là cách một contrast âm thầm được train dài hơn baseline mà nó bị đem
ra so; đó là bug thật của pipeline gốc (`max_steps=60` ở NB4 so với 30 step ở NB3).

**Bản Kaggle thêm `eval_dataset`.** Split `val` đã tồn tại từ §5 và trong repo không
ai dùng: 30 step trên ~225 mẫu ở LR 1e-4 là *đang thiếu* hay *đã quá* khớp thì cho
tới giờ vẫn là một lời phỏng đoán. Có loss held-out, câu đó thành số đo — và
`train.losses_from_log()` ghi lại cả hai đường cong.

In [ ]:
import time
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

ADAPTERS = WORK / "adapters"

# Ngân sách step, dẫn ra MỘT lần từ corpus của `correct`. `correct_replay` KHÔNG
# được tự tính từ độ dài corpus của nó: trộn replay làm corpus dài ra, và nếu để
# nó tự tính thì nó sẽ được train nhiều step hơn — thành hai biến thay vì một.
_rows_correct = data.to_training_dataset(tok, train_rows,
                                         max_length=TIER.max_length,
                                         mask_mode=MASK_MODE)
STEPS = train.planned_steps(len(_rows_correct), TIER, C.training_epochs())
print(f"epochs={C.training_epochs()}  ·  {len(_rows_correct)} mẫu  ·  "
      f"batch hiệu dụng {TIER.effective_batch}  ->  {STEPS} optimizer step")
print("Mọi run được chấm điểm dùng ĐÚNG con số này.")


def train_one(key, train_records, steps, *, val_records=None,
              replay_fraction=None):
    spec = C.SPECS[key]
    adir = ADAPTERS / key
    if (adir / "adapter_model.safetensors").exists() and not FORCE_RETRAIN:
        print(f"bỏ qua {key}: đã có {adir} (FORCE_RETRAIN=True để train lại)")
        return None

    print("=" * 78)
    print(f"RUN {key}: {spec.label}\n     {spec.teaches}")
    model, tk = generate.load_base(TIER, load_in_4bit=spec.load_in_4bit)
    targets = modeling.resolve_target_modules(model, spec.target)
    if spec.r is None:
        # attn_only: giải ra rank để BẰNG ngân sách tham số của `correct`. So
        # q,v @ r=16 với all-linear @ r=16 là so ngân sách, không phải so vị trí.
        base_t = modeling.resolve_target_modules(model, "text-linear")
        spec = spec.resolved(
            modeling.matched_rank(model, base_t, C.SPECS["correct"].r, targets))
        print(f"  rank khớp ngân sách: r={spec.r}  alpha={spec.alpha}")
    trainable = modeling.count_lora_params(model, targets, spec.r)
    print(f"  placement={spec.target}  modules={len(targets)}  "
          f"trainable ≈ {trainable/1e6:.2f} M  lr={spec.lr}  4bit={spec.load_in_4bit}")

    ds = Dataset.from_list(data.to_training_dataset(
        tk, train_records, max_length=TIER.max_length, mask_mode=MASK_MODE))
    val_ds = None
    if val_records:
        val_ds = Dataset.from_list(data.to_training_dataset(
            tk, val_records, max_length=TIER.max_length, mask_mode=MASK_MODE))
    sup = sum(sum(1 for x in r["labels"] if x != data.IGNORE_INDEX) for r in ds)
    tot = sum(len(r["labels"]) for r in ds)
    assert 0 < sup < tot, "mask không che gì hoặc che tất cả — quay lại §5"
    print(f"  train_ds={len(ds)}  val_ds={0 if val_ds is None else len(val_ds)}  "
          f"supervised {sup}/{tot} ({sup/tot:.1%})")

    want = train.sft_config_kwargs(
        TIER, spec, str(adir), max_steps=steps, mask_mode=MASK_MODE, seed=SEED,
        eval_steps=(max(1, steps // 6) if val_ds is not None else None))
    sft_kwargs, dropped = train.filter_kwargs(SFTConfig, want,
                                              label=f"SFTConfig[{key}]")
    if dropped:
        print("  ⚠ TRL không nhận:", dropped)
    lora_kwargs, _ = train.filter_kwargs(
        LoraConfig, train.lora_config_kwargs(spec, targets),
        label=f"LoraConfig[{key}]")

    generate.free_memory()
    trainer = SFTTrainer(model=model, args=SFTConfig(**sft_kwargs),
                         train_dataset=ds, eval_dataset=val_ds,
                         processing_class=tk,
                         peft_config=LoraConfig(**lora_kwargs))
    # TRL trả về trọng số LoRA ở bf16 bất kể thiết bị; GradScaler của fp16 không
    # có kernel BFloat16 nên run chết ở step 0. No-op trên bf16/fp32.
    fix = train.align_trainable_precision(trainer.model)
    if fix.get("recast"):
        print(f"  precision fix: {fix['recast']}/{fix['trainable_tensors']} "
              f"tensor bf16 -> fp32 cho GradScaler fp16")

    t0 = time.perf_counter()
    trainer.train()
    elapsed = time.perf_counter() - t0

    # LƯU TRƯỚC KHI LÀM GÌ KHÁC. Chấm điểm có thể OOM; adapter thì đã an toàn.
    trainer.model.save_pretrained(adir)
    tk.save_pretrained(adir)
    print(f"  saved -> {adir}")

    curves = train.losses_from_log(trainer.state.log_history)
    row = train.summarize_run(
        spec, TIER, targets, trainable, elapsed, generate.peak_vram_gb(),
        max_steps=steps, n_train_examples=len(ds),
        final_train_loss=curves["final_train_loss"],
        final_eval_loss=curves["final_eval_loss"],
        replay_fraction=replay_fraction)
    row["mask_mode"] = MASK_MODE
    row["teaches"] = spec.teaches
    report.append_row(row, results_dir=RESULTS)
    report.write_json(curves, f"curves_{key}.json", results_dir=RESULTS)
    print(f"  {elapsed:.0f}s  train_loss={curves['final_train_loss']}  "
          f"eval_loss={curves['final_eval_loss']}  "
          f"eval tăng sau đáy={curves['eval_rose_after_min']}")

    del trainer, model
    generate.free_memory()
    return row

In [ ]:
row_correct = train_one("correct", train_rows, STEPS, val_records=val_rows)
print(json.dumps(row_correct, ensure_ascii=False, indent=2)
      if row_correct else "(dùng adapter đã có từ lần chạy trước)")

### 7b. `correct_replay` — cùng LoRA, **khác dữ liệu** (deck §14.3)

Đây là đối chứng mà pipeline gốc còn thiếu, nhắm vào đúng khuyết điểm mà chính repo
đo được và để mở: `regression` **0.644 → 0.067**. Mọi nút LoRA giống `correct` từng
con số; biến duy nhất là corpus, và số step vẫn là `STEPS` của `correct`.

Nếu run này giữ được regression mà không mất target, thì kết luận của lab không còn
là *"fine-tune làm model quên"* mà là *"corpus chỉ-một-tác-vụ làm model quên, và
deck §14.3 chữa được bằng 5% dữ liệu"* — hai câu khác nhau hoàn toàn về hành động.

In [ ]:
row_replay = None
if RUN_REPLAY:
    row_replay = train_one("correct_replay", mixed_rows, STEPS,
                           val_records=val_rows,
                           replay_fraction=C.replay_fraction())
    print(json.dumps(row_replay, ensure_ascii=False, indent=2)
          if row_replay else "(dùng adapter đã có)")
else:
    print("RUN_REPLAY=False — bỏ §7b. Phán quyết sẽ chỉ có `correct`.")

## 8. Ba cấu hình SAI — cùng số step, mỗi lần đổi đúng một biến

Bản Day-21 *cũ* lấy "quét rank r=8/16/64", gắn adapter vào `q_proj,v_proj`, và chấm
bằng perplexity làm thí nghiệm trung tâm. Deck hiện tại gọi đúng ba thứ đó là **Lỗi
#1, #2, #3** (§10.2–§10.4). Phần này chạy lại thí nghiệm cũ **như một đối chứng**.

| Run | Đổi gì | Kỳ vọng |
|---|---|---|
| `attn_only` | chỉ q,v — **rank nâng lên cho BẰNG số tham số** | thua `correct` |
| `wrong_lr` | LR thang full-FT (÷10) | loss gần như phẳng |
| `qlora` | 4-bit thay 16-bit | nhẹ hơn, chất lượng ? |

Bảng dưới in `final_train_loss`. **Đừng xếp hạng bằng cột đó** — làm vậy chính là Lỗi
#3. §9 chấm cả ba adapter này trên tập target, bằng đúng thang đo đã dùng cho
`correct`; nếu thứ tự hai bảng khác nhau thì bạn vừa tự tay đo được lý do lab cũ kết
luận sai.

In [ ]:
if RUN_CONTRASTS:
    _m, _t = generate.load_base(TIER)
    print(report.markdown_table(modeling.describe_placement(_m, C.SPECS["correct"].r)))
    del _m
    generate.free_memory()

    for key in C.CONTRAST_KEYS:
        train_one(key, train_rows, STEPS, val_records=val_rows)
else:
    print("RUN_CONTRASTS=False — bỏ §8. §9 sẽ không có phần giải phẫu.")

_seen = {}
for r in report.read_rows("runs.csv", results_dir=RESULTS):
    if r.get("run"):
        _seen[r["run"]] = r          # dòng cuối của mỗi key là dòng hiện hành
runs_rows = [_seen[k] for k in C.GRADED_KEYS if k in _seen]
cols = ["run", "r", "lora_alpha", "learning_rate", "load_in_4bit",
        "trainable_params", "max_steps", "n_train_examples", "replay_fraction",
        "final_train_loss", "final_eval_loss", "train_seconds", "peak_vram_gb"]
print()
print(report.markdown_table(runs_rows, cols))

budgets = {r["run"]: r.get("max_steps") for r in runs_rows}
if len(set(budgets.values())) > 1:
    print(f"\n⚠ ngân sách step KHÔNG bằng nhau: {budgets} — bảng so sánh này "
          "đang đo độ dài train, không phải cấu hình.")
else:
    print(f"\ncả {len(budgets)} run cùng {next(iter(budgets.values()))} step — "
          "so sánh công bằng.")

## 9. Bốn nhóm điểm cho MỌI phiên bản, và **phán quyết**

Fine-tune được chấm với **prompt ngây thơ**, không phải prompt (b): cái mà fine-tune
mua cho bạn chính là "hành vi đã chuyển vào trọng số nên prompt co lại được". Đó cũng
là điều làm phần cost ở §10 có ý nghĩa.

(Và nó chỉ đúng vì §5.3 đã kiểm: chuỗi mà evaluation gửi đi là **tiền tố** của chuỗi
đã train. Khi hai thứ đó lệch nhau — lỗi F-31 — mọi adapter ra `target = 0.000`.)

In [ ]:
from peft import PeftModel

def score_adapter(key, version, *, full=True):
    spec = C.SPECS[key]
    adir = ADAPTERS / key
    if not (adir / "adapter_model.safetensors").exists():
        print(f"bỏ qua {version}: chưa có {adir}")
        return None
    # load_in_4bit phải KHỚP lúc train: chấm adapter `qlora` trên base 16-bit là
    # đo độ lệch base/adapter rồi gọi nó là "giá của QLoRA".
    model, tk = generate.load_base(TIER, load_in_4bit=spec.load_in_4bit)
    model = PeftModel.from_pretrained(model, str(adir))
    model.eval()
    sc = eval_pass(model, tk, version, C.NAIVE_PROMPT, full=full)
    del model
    generate.free_memory()
    return sc

score_adapter("correct", V_C)
if RUN_REPLAY and (ADAPTERS / "correct_replay" / "adapter_model.safetensors").exists():
    score_adapter("correct_replay", V_CR)

### 9.1 Bảng bốn nhóm + biểu đồ, cho mọi phiên bản

In [ ]:
order = [v for v in (V_A, V_B, V_C, V_CR) if v in SCORES]
table = ev.comparison_table({v: SCORES[v] for v in order})
print(report.markdown_table(table))

for group in ("target", "regression", "format"):
    print(f"\n{group}:")
    print(report.hbar({v: getattr(SCORES[v], group) for v in order}, vmax=1.0))
print("\nlatency (ms/mẫu, thấp hơn là tốt hơn):")
print(report.hbar({v: SCORES[v].latency_ms for v in order}, fmt="{:.0f}"))

print("\nđộ chính xác từng field trên tập target:")
field_tbl = [{"version": v, **{k: (None if x is None else round(x, 3))
                               for k, x in SCORES[v].extra["fields"].items()}}
             for v in order]
print(report.markdown_table(field_tbl))

if HOLD:
    print("\nholdout_secret (20 ticket chưa dùng ở bất kỳ đâu):")
    print(report.hbar({v: HOLD[v]["target"] for v in order if v in HOLD}, vmax=1.0))

### 9.2 Cổng hồi quy — chấm **cả hai** bản fine-tune so với (b)

Đạt = vượt (b) ở `target` **và** không tụt `regression` quá 0.02. Một phán quyết
**FAILED được chấm điểm đầy đủ** nếu bạn phân tích trung thực; cái bị trừ điểm là nới
ngưỡng, làm yếu prompt (b), hay đổi tập eval sau khi đã thấy kết quả.

In [ ]:
verdicts = {}
for v in (V_C, V_CR):
    if v not in SCORES:
        continue
    verd = ev.regression_gate(SCORES[v], SCORES[V_B])
    verdicts[v] = verd
    print("=" * 78)
    print(f"{v}: {'PASSED' if verd.passed else 'FAILED'}")
    for r in verd.reasons:
        print("  -", r)

winner = None
cands = [v for v in (V_C, V_CR) if v in verdicts]
passed = [v for v in cands if verdicts[v].passed]
if passed:
    winner = max(passed, key=lambda v: SCORES[v].target)
elif cands:
    # Không ai qua cổng: "người thắng" là bản đánh đổi ít nhất, và §10/§13 nói
    # thẳng rằng nó KHÔNG qua cổng.
    winner = max(cands, key=lambda v: SCORES[v].target + SCORES[v].regression)
print("\n" + "=" * 78)
print(f"bản chọn để so cost/serve: {winner}  "
      f"(qua cổng: {bool(passed)})")

if V_C in SCORES and V_CR in SCORES:
    dt = SCORES[V_CR].target - SCORES[V_C].target
    dr = SCORES[V_CR].regression - SCORES[V_C].regression
    print(f"\nreplay mix {C.replay_fraction():.0%} đổi được: regression {dr:+.3f}, "
          f"target {dt:+.3f} — deck §14.3, một biến duy nhất là DỮ LIỆU.")

### 9.3 Giải phẫu: ba cấu hình sai, chấm trên **thang đo tác vụ**

In [ ]:
autopsy = [{"run": k, "version": v, "target": round(SCORES[v].target, 4),
            "format": round(SCORES[v].format, 4),
            "latency_ms": round(SCORES[v].latency_ms, 1), "n": SCORES[v].n}
           for k, v in (("correct", V_C), ("correct_replay", V_CR))
           if v in SCORES]
for key in C.CONTRAST_KEYS:
    sc = score_adapter(key, key, full=False)
    if sc is None:
        continue
    autopsy.append({"run": key, "version": key, "target": round(sc.target, 4),
                    "format": round(sc.format, 4),
                    "latency_ms": round(sc.latency_ms, 1), "n": sc.n})
print()
print(report.markdown_table(autopsy, ["run", "target", "format", "latency_ms", "n"]))
report.write_json(autopsy, "autopsy.json", results_dir=RESULTS)

loss_order = [r["run"] for r in sorted(
    (r for r in runs_rows if str(r.get("final_train_loss", "")).strip()),
    key=lambda r: float(r["final_train_loss"]))]
task_order = [r["run"] for r in sorted(autopsy, key=lambda r: -r["target"])]
print(f"\nxếp theo train loss (thấp->cao):  {loss_order}")
print(f"xếp theo điểm target (cao->thấp): {task_order}")
if loss_order and task_order and loss_order[0] != task_order[0]:
    print("\n→ Hai thứ tự KHÁC nhau. Đây chính là Lỗi #3 hiện ra thành số: xếp "
          "hạng bằng loss huấn luyện sẽ chọn sai run.")

### 9.4 Định tính — **bắt buộc có cả ca THUA**

Chọn ca thắng thôi là cherry-pick và bị trừ ở mục Evaluation Quality. Ba ca tệ nhất
ở dưới là ba ca phải xuất hiện trong REPORT.md.

In [ ]:
assert winner, ("chưa có adapter nào được chấm — chạy §7 (và §9 cell đầu) trước "
                "khi đọc phần định tính.")
qual = []
for i, (p, r) in enumerate(zip(PREDS[winner]["target"], target)):
    base_p = PREDS[V_B]["target"][i]
    qual.append({
        "i": i,
        "ticket": r["input"][:64],
        "ft": round(ev.triage_field_accuracy(p, r["label"]), 2),
        "b": round(ev.triage_field_accuracy(base_p, r["label"]), 2),
        "ft_pred": p.replace("\n", " ")[:80],
        "b_pred": base_p.replace("\n", " ")[:80],
    })
qual.sort(key=lambda x: (x["ft"] - x["b"], x["ft"]))
cols_q = ["i", "ticket", "b", "ft", "ft_pred"]
print("--- 3 ca fine-tune THUA/kém nhất so với (b) ---")
print(report.markdown_table(qual[:3], cols_q))
print("\n--- 3 ca fine-tune THẮNG rõ nhất ---")
print(report.markdown_table(qual[-3:], cols_q))
report.write_json(qual, "qualitative.json", results_dir=RESULTS)

# Nhóm regression cũng cần một ca đọc bằng mắt: điểm keyword_recall = 0 có thể là
# "trả lời sai" hoặc là "trả lời bằng JSON triage" — hai chuyện khác nhau, và chỉ
# cái thứ hai mới là quên thảm hoạ.
print("\n--- regression: cùng một câu hỏi, ba phiên bản ---")
q0 = regression[0]
print(f"hỏi: {q0['instruction']}   (keywords={q0['keywords']})")
for v in order:
    if PREDS[v]["regression"]:
        print(f"  [{v}] -> {PREDS[v]['regression'][0].replace(chr(10), ' ')[:150]}")

In [ ]:
payload = {
    "comparison": table,
    "verdict": verdicts[winner].as_dict() if winner in verdicts else None,
    "verdicts": {v: verdicts[v].as_dict() for v in verdicts},
    "winner": winner,
    "passed_gate": winner in passed,
    "regression_tolerance": ev.REGRESSION_TOLERANCE,
    "holdout": HOLD,
    "field_accuracy": {v: SCORES[v].extra["fields"] for v in order},
    "valid_trace_rate": {v: SCORES[v].extra.get("valid_trace_rate") for v in order},
    "n_target": len(target), "n_regression": len(regression),
    "smoke_mode": bool(LIMIT) or TIER.name == "SMOKE",
}
report.write_json(payload, "verdict.json", results_dir=RESULTS)
print(json.dumps({k: payload[k] for k in
                  ("winner", "passed_gate", "verdict", "smoke_mode")},
                 ensure_ascii=False, indent=2))

## 10. Latency và **cost** — nửa còn lại của câu "có nên fine-tune?"

§9 chọn ra bản thắng về chất lượng. Một team còn phải trả lời câu thứ hai: *nó tốn
bao nhiêu, và bao lâu thì hoàn vốn?* Fine-tune trả trước bằng GPU-giờ và mua lại
**prompt ngắn hơn** cùng **output ngắn hơn** trên mọi request, mãi mãi.

Tất cả số ở đây là **phép tính trên số đo**: `mean_latency_ms` (đã trừ warm-up),
`prompt_tokens_mean`, `new_tokens_mean` đều do `generate_measured()` đo ở §6/§9. Giá
thì là **giả định có nguồn** và là tham số — sửa ở ô CONFIG rồi chạy lại ô này.

Ba cảnh báo đi kèm, nói một lần và nói thật:
1. `mean_latency_ms` là **throughput theo batch** ở batch size của vòng chấm. Phục vụ
   một ticket lẻ thì chậm hơn; batch 32 thì nhanh hơn. So sánh **giữa các phiên bản**
   là công bằng vì mọi phiên bản đo cùng batch size — nhưng đây không phải báo giá prod.
2. Cost self-host giả định GPU **luôn có việc**. GPU thuê mà để không vẫn tính tiền,
   nên đây là mức sàn chỉ đạt được khi hàng đợi luôn đầy.
3. Không có dòng nào tính tiền công engineer, và bản fine-tune đã tiêu một ít.

In [ ]:
key_of = {V_C: "correct", V_CR: "correct_replay"}
train_seconds = None
if winner in key_of and key_of[winner] in _seen:
    raw = _seen[key_of[winner]].get("train_seconds")
    train_seconds = float(raw) if str(raw).strip() else None

comp = cost.compare(
    {v: STATS[v] for v in order},
    train_seconds=train_seconds, baseline=V_B, winner=winner,
    gpu_hourly_usd=GPU_HOURLY_USD,
    input_usd_per_mtok=API_IN_USD_MTOK,
    output_usd_per_mtok=API_OUT_USD_MTOK)

print(report.markdown_table(comp["rows"]))
print()
print(json.dumps(comp["delta"], ensure_ascii=False, indent=2))
print("\n" + cost.verdict_line(comp))

# Chi phí thật của cả buổi lab, không chỉ của run thắng: hoá đơn GPU tính theo
# số giờ đã chạy, kể cả những run chỉ để chứng minh cấu hình sai là sai.
all_seconds = sum(float(r["train_seconds"]) for r in runs_rows
                  if str(r.get("train_seconds", "")).strip())
print(f"\ntổng thời gian train của MỌI run: {all_seconds/60:.0f} phút "
      f"= ${cost.training_usd(all_seconds, GPU_HOURLY_USD):.3f} "
      f"(giá {GPU_HOURLY_USD}/GPU-giờ)")
if winner not in passed:
    print("⚠ Bản này KHÔNG qua cổng hồi quy ở §9. Con số break-even ở trên chỉ "
          "trả lời 'rẻ hơn khi nào', không trả lời 'có nên ship'.")
report.write_json({**comp, "all_runs_train_seconds": round(all_seconds, 1),
                   "all_runs_train_usd": round(cost.training_usd(all_seconds,
                                                                 GPU_HOURLY_USD), 4),
                   "winner": winner, "winner_passed_gate": winner in passed},
                  "cost.json", results_dir=RESULTS)

## 11. Chạy thử — cùng câu hỏi, mọi phiên bản, đặt cạnh nhau

Bảng điểm nói *bao nhiêu*; phần này nói *khác nhau ở đâu*. Sáu câu đã được sinh sẵn
trong chính lượt nạp model của mỗi phiên bản ở §6/§9, nên ở đây không tốn thêm GPU.

Ba ticket: một ca thẳng, một ca **hai ý xung đột** (giao chậm *và* hàng lỗi *và* xin
hoàn tiền — chỗ mà `intent` buộc phải chọn), và một ca **ngoài miền** (không phải
ticket khiếu nại; model nên làm gì với nó?).

Ba câu tổng quát: đây là chỗ **quên thảm hoạ** hiện ra bằng mắt. Nếu bản fine-tune
trả lời "Thủ đô của Nhật Bản là thành phố nào?" bằng một object JSON triage, bạn
không cần bảng điểm nào để biết đã xảy ra chuyện gì.

In [ ]:
for i, (kind, ticket) in enumerate(SAMPLE_TICKETS):
    print("=" * 78)
    print(f"TICKET [{kind}] {ticket}")
    for v in order:
        out = PREDS[v]["samples"]
        if out:
            print(f"\n  [{v}]\n    {out[i].replace(chr(10), ' ')[:300]}")
    print()

In [ ]:
for i, q in enumerate(SAMPLE_GENERAL):
    print("=" * 78)
    print(f"CÂU HỎI THƯỜNG NGÀY: {q}")
    for v in order:
        out = PREDS[v]["general"]
        if out:
            print(f"\n  [{v}]\n    {out[i].replace(chr(10), ' ')[:300]}")
    print()

report.write_json(
    {"tickets": [{"kind": k, "prompt": t,
                  "answers": {v: (PREDS[v]["samples"][i] if PREDS[v]["samples"] else None)
                              for v in order}}
                 for i, (k, t) in enumerate(SAMPLE_TICKETS)],
     "general": [{"prompt": q,
                  "answers": {v: (PREDS[v]["general"][i] if PREDS[v]["general"] else None)
                              for v in order}}
                 for i, q in enumerate(SAMPLE_GENERAL)]},
    "samples.json", results_dir=RESULTS)

## 12. (Tuỳ chọn) Merge, kiểm chứng sau merge, và hoán đổi adapter — deck §18

Hai đường triển khai:
* **Merge** — `W = W₀ + (α/r)·BA`; đồ thị phục vụ giống hệt base → **không** overhead.
* **Giữ riêng** — một base trong VRAM, nhiều adapter, chọn theo từng request.

`RUN_MERGE=False` theo mặc định vì merge ghi ~8 GB xuống `/kaggle/working` và Kaggle
giới hạn 20 GB output. Phần hoán đổi adapter thì rẻ và luôn chạy.

**Assert bắt buộc:** merge là phép toán chính xác trên giấy, nhưng dtype lúc gộp có
thể làm tụt điểm. Tụt thì **đừng deploy** — đi tìm nguyên nhân.

In [ ]:
from peft import PeftModel

base_key = key_of.get(winner, "correct")
model, tk = generate.load_base(TIER)
model = PeftModel.from_pretrained(model, str(ADAPTERS / base_key),
                                 adapter_name=base_key)
loaded = [base_key]
for extra in [k for k in C.GRADED_KEYS if k != base_key]:
    d = ADAPTERS / extra
    if (d / "adapter_model.safetensors").exists():
        model.load_adapter(str(d), adapter_name=extra)
        loaded.append(extra)
print("một base trong VRAM, các adapter đã nạp:", loaded)
print(f"VRAM đang dùng: {generate.peak_vram_gb():.2f} GB")

ticket = target[0]["input"]
print(f"\nticket: {ticket[:120]}\nnhãn đúng: {target[0]['label']}\n")
for name in loaded:
    model.set_adapter(name)
    out, _ = generate.generate_measured(model, tk, [ticket],
                                        system=C.NAIVE_PROMPT, max_new_tokens=96,
                                        batch_size=1, warmup=False, progress=False)
    print(f"  [{name}] -> {out[0].replace(chr(10), ' ')[:160]}")
model.set_adapter(base_key)

In [ ]:
merge_check = None
if RUN_MERGE:
    n_chk = min(len(target), 20)
    chk = target[:n_chk]
    pre, _ = generate.generate_measured(
        model, tk, [r["input"] for r in chk], system=C.NAIVE_PROMPT,
        max_new_tokens=MAX_NEW, batch_size=EVAL_BATCH, label="pre-merge")
    before = sum(ev.triage_field_accuracy(p, r["label"])
                 for p, r in zip(pre, chk)) / n_chk

    merged = model.merge_and_unload()
    post, _ = generate.generate_measured(
        merged, tk, [r["input"] for r in chk], system=C.NAIVE_PROMPT,
        max_new_tokens=MAX_NEW, batch_size=EVAL_BATCH, label="post-merge")
    after = sum(ev.triage_field_accuracy(p, r["label"])
                for p, r in zip(post, chk)) / n_chk
    TOL = 0.01
    merge_check = {"adapter": base_key, "before_merge": round(before, 4),
                   "after_merge": round(after, 4), "delta": round(after - before, 4),
                   "tolerance": TOL, "n": n_chk}
    print(json.dumps(merge_check, ensure_ascii=False, indent=2))
    assert after - before >= -TOL, (
        f"điểm TỤT {before - after:.4f} sau merge (ngưỡng {TOL}). Kiểm tra dtype "
        "lúc gộp; với DoRA cần PEFT >= 0.10 để gộp đúng magnitude vector (§18).")
    merged.save_pretrained(ADAPTERS / "merged")
    tk.save_pretrained(ADAPTERS / "merged")
    report.write_json(merge_check, "merge_check.json", results_dir=RESULTS)
    del merged
else:
    print("RUN_MERGE=False — bỏ phần merge (ghi ~8 GB). Phần hot-swap ở trên đã "
          "chứng minh đường triển khai 'một base, nhiều adapter'.")

del model
generate.free_memory()

## 13. Cổng kiểm tra trước khi nộp

Ô này là `scripts/verify.py` của repo, viết thẳng vào notebook (luật: không clone).
Nó **không** chấm điểm bài — nó kiểm những điều kiện mà nếu sai thì điểm không có
nghĩa gì:

| kiểm | hỏng thì sao |
|---|---|
| corpus khớp checksum | sửa tập eval sau khi thấy điểm → mọi so sánh vô nghĩa |
| không ở SMOKE mode | số của tier 0.8B / EVAL_LIMIT không phải bài nộp |
| mask đúng | supervise cả câu hỏi → model học đoán ticket, không học trả lời |
| prompt train ≡ prompt eval | F-31: mọi adapter chấm 0.000 dù train tốt |
| replay đã tẩy trùng | replay chứa câu trong tập regression → điểm hồi quy là rò rỉ |
| mọi run cùng ngân sách step | bảng contrast đo độ dài train, không đo cấu hình |
| `attn_only` khớp tham số ±5% | contrast placement biến thành contrast dung lượng |
| có phán quyết + bảng cost | thiếu chính phần kết luận |

In [ ]:
import zipfile

checks = []
def chk(name, ok, detail=""):
    checks.append({"check": name, "pass": bool(ok), "detail": detail})
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f" — {detail}" if detail else ""))

print("=" * 78)
chk("corpus khớp checksum (CRLF-tolerant)", not integrity["drift"],
    f"{len(integrity['files'])} file, drift={integrity['drift'] or 'không'}")
chk("không chạy ở SMOKE mode", not SMOKE,
    f"tier={TIER.name}, EVAL_LIMIT={LIMIT or 'FULL'}")
chk("eval target dùng đủ bộ", len(target) >= 50, f"n={len(target)}")
chk("eval regression dùng đủ bộ", len(regression) >= 15, f"n={len(regression)}")
chk("holdout được chấm", bool(HOLD), f"n={len(holdout)}, versions={len(HOLD)}")

chk("mask: câu trả lời nằm trong loss", proof["answer_is_supervised"])
chk("mask: câu hỏi bị che", proof["question_is_masked"],
    f"supervised_fraction={proof['supervised_fraction']:.3f}")
chk("prompt train ≡ prompt eval (F-31)", align["all_aligned"],
    f"n_checked={align['n_checked']}")
chk("replay đã tẩy trùng", decon["exact_collisions"] == 0
    and decon["worst_jaccard"] <= decon["jaccard_threshold"],
    f"worst_jaccard={decon['worst_jaccard']} (ngưỡng {decon['jaccard_threshold']})")

In [ ]:
budget_vals = {r["run"]: r.get("max_steps") for r in runs_rows}
chk("mọi run cùng ngân sách optimizer step", len(set(budget_vals.values())) == 1,
    str(budget_vals))

def _p(row, k):
    v = str(row.get(k, "")).strip()
    return float(v) if v else None
_cor = _seen.get("correct")
_att = _seen.get("attn_only")
if _cor and _att:
    a, b = _p(_cor, "trainable_params"), _p(_att, "trainable_params")
    rel = abs(b - a) / a if a else 1.0
    chk("attn_only khớp tham số ±5%", rel <= 0.05,
        f"{int(a):,} vs {int(b):,} ({rel:+.2%})")
else:
    chk("attn_only khớp tham số ±5%", False, "chưa chạy §8 (RUN_CONTRASTS=False)")

chk("có phán quyết cho ít nhất một fine-tune", bool(verdicts),
    ", ".join(f"{v}={'PASS' if verdicts[v].passed else 'FAIL'}" for v in verdicts))
chk("fine-tune thắng (b) và giữ được năng lực chung", winner in passed,
    (verdicts[winner].reasons[0][:110] if winner in verdicts else "không có"))
chk("có bảng cost + break-even", "delta" in comp,
    f"self-host saving {comp.get('delta', {}).get('self_host_saving_per_1k')} $/1k")

REQUIRED = ["template_check.json", "mask_proof.json", "prompt_alignment.json",
            "token_stats.json", "replay_manifest.json", "baselines_frozen.json",
            "runs.csv", "curves_correct.json", "autopsy.json", "qualitative.json",
            "verdict.json", "cost.json", "samples.json"]
missing = [f for f in REQUIRED if not (RESULTS / f).exists()]
chk("đủ artifact bắt buộc", not missing, f"thiếu: {missing or 'không'}")

n_fail = sum(1 for c in checks if not c["pass"])
print("=" * 78)
print(f"{len(checks) - n_fail}/{len(checks)} PASS" +
      ("  ✅ sẵn sàng nộp" if not n_fail else f"  ❌ {n_fail} mục cần sửa"))

### 13.1 `run_config.json` — chạy lại được thì mới gọi là kết quả

Một bảng điểm không kèm cấu hình sinh ra nó là một tin nhắn, không phải phép đo.

In [ ]:
run_config = {
    "tier": TIER.name, "model_id": TIER.model_id,
    "precision": device.precision(),
    "max_length": TIER.max_length,
    "per_device_batch": TIER.per_device_batch,
    "grad_accum": TIER.grad_accum,
    "effective_batch": TIER.effective_batch,
    "epochs": C.training_epochs(), "planned_steps": STEPS,
    "mask_mode": MASK_MODE, "seed": SEED,
    "replay_fraction": C.replay_fraction(),
    "eval_limit": LIMIT or None, "smoke_mode": SMOKE,
    "eval_batch": EVAL_BATCH, "max_new_tokens": MAX_NEW,
    "n_train": len(train_rows), "n_val": len(val_rows),
    "n_target": len(target), "n_regression": len(regression),
    "n_holdout": len(holdout),
    "specs": {k: {"target": C.SPECS[k].target, "r": C.SPECS[k].r,
                  "alpha": C.SPECS[k].alpha, "lr": C.SPECS[k].lr,
                  "load_in_4bit": C.SPECS[k].load_in_4bit}
              for k in C.GRADED_KEYS if k in C.SPECS},
    "prices": {"gpu_hourly_usd": GPU_HOURLY_USD,
               "api_in_usd_mtok": API_IN_USD_MTOK,
               "api_out_usd_mtok": API_OUT_USD_MTOK},
    "integrity": integrity,
    "gate": {"checks": checks, "n_fail": n_fail},
    "versions": {},
}
import torch, transformers, trl, peft, datasets as _ds
run_config["versions"] = {"torch": torch.__version__,
                          "transformers": transformers.__version__,
                          "trl": trl.__version__, "peft": peft.__version__,
                          "datasets": _ds.__version__}
report.write_json(run_config, "run_config.json", results_dir=RESULTS)
print(json.dumps({k: run_config[k] for k in
                  ("tier", "precision", "epochs", "planned_steps", "mask_mode",
                   "smoke_mode", "versions")}, ensure_ascii=False, indent=2))

### 13.2 `REPORT.md` — bản nháp đã điền **số đo thật**

Ô này không viết hộ phần kết luận. Nó điền mọi con số vào chỗ của nó rồi để lại
những câu hỏi mà chỉ người chạy trả lời được (vì sao cấu hình sai thua, bạn sẽ ship
bản nào). Sửa trong `/kaggle/working/REPORT.md` rồi tải về.

In [ ]:
def _f(x, nd=4):
    return "—" if x is None else f"{x:.{nd}f}"

d = comp.get("delta", {})
fields_tbl = report.markdown_table(
    [{"version": v, **{k: _f(SCORES[v].extra["fields"].get(k), 3)
                       for k in ev.TRIAGE_KEYS},
      "unparsed": SCORES[v].extra["fields"]["unparsed"]} for v in order])

lines = [
    "# Day-21 Track-3 — LoRA fine-tune cho phân loại ticket tiếng Việt",
    "",
    f"- model: `{TIER.model_id}` · tier `{TIER.name}` · precision "
    f"`{device.precision()}`",
    f"- LoRA: r={C.SPECS['correct'].r}, alpha={C.SPECS['correct'].alpha}, "
    f"lr={C.SPECS['correct'].lr}, placement=`{C.SPECS['correct'].target}`",
    f"- ngân sách: {STEPS} optimizer step ({C.training_epochs()} epoch, effective "
    f"batch {TIER.effective_batch}) — **giống nhau cho mọi run**",
    f"- mask: `{MASK_MODE}` · {proof['supervised_fraction']:.1%} token vào loss",
    f"- eval: target n={len(target)}, regression n={len(regression)}, "
    f"holdout n={len(holdout)}" + ("  ⚠ **SMOKE MODE**" if SMOKE else ""),
    "",
    "## 1. Bảng bốn nhóm, ba (bốn) phiên bản",
    "",
    report.markdown_table(table),
    "",
    "Cột `target` là accuracy trung bình trên 4 field triage; `regression` là "
    "keyword-recall trên 15 câu hỏi tổng quát; `format` là tỉ lệ output có đúng "
    "4 key; `latency_ms` là throughput theo batch, đã trừ warm-up.",
    "",
    f"- (b) − (a) = **{SCORES[V_B].target - SCORES[V_A].target:+.3f}** target — "
    "phần này prompt engineering làm được, **không** cần fine-tune.",
    f"- {winner} − (b) = **{SCORES[winner].target - SCORES[V_B].target:+.3f}** "
    f"target, **{SCORES[winner].regression - SCORES[V_B].regression:+.3f}** "
    "regression — phần này mới là công của fine-tune.",
    f"- cổng hồi quy (dung sai {ev.REGRESSION_TOLERANCE:.2f}): "
    f"**{'PASS' if winner in passed else 'FAIL'}**",
    "",
]
for v in verdicts:
    lines += [f"**{v}** — {'PASSED' if verdicts[v].passed else 'FAILED'}"] + \
             [f"  - {r}" for r in verdicts[v].reasons] + [""]

In [ ]:
lines += [
    "## 2. Autopsy per-field",
    "",
    fields_tbl,
    "",
    "> TODO: field nào tụt nhiều nhất, và vì sao? `unparsed` > 0 nghĩa là mất điểm "
    "vì **định dạng**, không phải vì phân loại sai — hai lỗi khác nhau.",
    "",
    "## 3. Cấu hình sai, chấm trên thang đo tác vụ",
    "",
    report.markdown_table(autopsy),
    "",
    f"Thứ tự theo train-loss: `{' < '.join(loss_order)}`",
    f"Thứ tự theo target:     `{' > '.join(task_order)}`",
    "",
    "> TODO: hai thứ tự có khác nhau không? Nếu có, đó chính là Lỗi #3 (xếp hạng "
    "cấu hình bằng loss) đo được trên chính run của bạn.",
    "",
    "## 4. Latency và cost",
    "",
    report.markdown_table(comp["rows"]),
    "",
    cost.verdict_line(comp),
    "",
    f"Giả định: ${GPU_HOURLY_USD}/GPU-giờ, API ${API_IN_USD_MTOK}/"
    f"${API_OUT_USD_MTOK} per Mtok. " + comp["assumptions"]["note"],
    "",
    f"Tổng train mọi run: {all_seconds/60:.0f} phút "
    f"≈ ${cost.training_usd(all_seconds, GPU_HOURLY_USD):.3f}.",
    "",
    "## 5. Holdout (không dùng để chọn bất cứ thứ gì)",
    "",
    report.markdown_table([{"version": v, "target": _f(HOLD[v]["target"], 4),
                            "n": HOLD[v]["n"]} for v in HOLD]) if HOLD
    else "_(không chấm holdout)_",
    "",
    "> TODO: điểm holdout có sát điểm target không? Lệch nhiều = tập target đã bị "
    "nhìn quá nhiều lần trong lúc chỉnh.",
    "",
    "## 6. Kết luận",
    "",
    f"- Cổng kiểm tra: {len(checks) - n_fail}/{len(checks)} PASS.",
    "- TODO: bạn sẽ ship bản nào, và **vì sao**? Nếu (b) đủ tốt thì câu trả lời "
    "đúng có thể là 'không fine-tune' — deck §17.",
    "- TODO: break-even ở trên có nằm trong lưu lượng thật của hệ thống bạn không?",
    "",
]
REPORT = WORK / "REPORT.md"
REPORT.write_text("\n".join(lines), encoding="utf-8")
print(f"đã ghi {REPORT}  ({len(lines)} dòng)")
print("\n".join(lines[:34]))

### 13.3 Đóng gói

Zip chỉ chứa `results/`, `REPORT.md` và các adapter — **không** chứa base model
(~8 GB) và không chứa lại corpus (đã có trong dataset gốc).

In [ ]:
SUB = WORK / "submission"
SUB.mkdir(exist_ok=True)
zpath = SUB / f"lab21_{TIER.name.lower()}{'_smoke' if SMOKE else ''}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(WORK))
    if (WORK / "REPORT.md").exists():
        z.write(WORK / "REPORT.md", "REPORT.md")
    for adir in sorted(ADAPTERS.glob("*")):
        if adir.name == "merged" or not adir.is_dir():
            continue
        for p in sorted(adir.rglob("*")):
            if p.is_file() and p.stat().st_size < 200 * 1024 * 1024:
                z.write(p, p.relative_to(WORK))
print(f"{zpath}  ({zpath.stat().st_size/1024**2:.1f} MB)")
for name in sorted(zipfile.ZipFile(zpath).namelist()):
    print("  ", name)
if n_fail:
    print(f"\n❌ {n_fail} mục FAIL ở §13 — xem lại trước khi nộp.")
else:
    print("\n✅ Toàn bộ cổng kiểm tra PASS. Tải zip ở tab Output.")